# Data10: single-tone frequency discrimination

Twenty-seven IM recordings cover nine settings around 50, 100 and 200 Hz, with three recordings per setting. The primary relative-frequency result supports Figure 2; absolute errors and alternative estimators remain available.


## Setup

Use a Python 3.11+ kernel with the release requirements installed. Run from the release folder.


In [ ]:
# sinewave-cell: setup
from pathlib import Path
from copy import deepcopy
from itertools import combinations
from hashlib import sha256
import json
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

from portable_runtime import (
    project_root, data_root, load_config, ensure_environment,
    notebook_path, source_paths, figure_font, config_source,
)

PROJECT = project_root()
ensure_environment()
NOTEBOOK_PATH = notebook_path("data10_resolution_analysis")
LOCAL_CONFIG = load_config()
RESULTS_ROOT = Path(LOCAL_CONFIG["results_root"]).expanduser().resolve()
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from characterization_run import AnalysisRun

pd.set_option("display.max_columns", 24)
pd.set_option("display.precision", 8)
print("Python:", sys.executable)
print("Results root:", RESULTS_ROOT)

# Reload the shared paper figure style for this Run All.
import importlib as _paper_importlib
import paper_figure_style as _paper_style
_paper_importlib.reload(_paper_style)


## Configuration

The explicit FILES list contains all 27 selected recordings. Main windows are [0,100) s; trimmed comparisons are [5,95) s. The primary estimator is global_padded_3point. Original-time sine fitting searches near the measured FFT peak, not nominal frequency. The legacy desired_records_per_setting=2 value is retained unchanged as an advisory field; it does not select or discard files, and actual configured/valid counts are reported.


In [ ]:
# sinewave-cell: configuration
from pathlib import Path

# Explicit selected inputs: 27 recordings, three per nominal setting.
# Actual selection is defined by FILES; all original recordings remain read only.
FILES = {'resolution': [{'path': 'IM_17.csv', 'target_hz': 50.0, 'base_hz': 50.0},
                {'path': 'IM_18.csv', 'target_hz': 50.0, 'base_hz': 50.0},
                {'path': 'IM_19.csv', 'target_hz': 50.0, 'base_hz': 50.0},
                {'path': 'IM_20.csv', 'target_hz': 50.01, 'base_hz': 50.0},
                {'path': 'IM_21.csv', 'target_hz': 50.01, 'base_hz': 50.0},
                {'path': 'IM_22.csv', 'target_hz': 50.01, 'base_hz': 50.0},
                {'path': 'IM_23.csv', 'target_hz': 50.005, 'base_hz': 50.0},
                {'path': 'IM_24.csv', 'target_hz': 50.005, 'base_hz': 50.0},
                {'path': 'IM_25.csv', 'target_hz': 50.005, 'base_hz': 50.0},
                {'path': 'IM_26.csv', 'target_hz': 100.0, 'base_hz': 100.0},
                {'path': 'IM_39.csv', 'target_hz': 100.0, 'base_hz': 100.0},
                {'path': 'IM_45.csv', 'target_hz': 100.0, 'base_hz': 100.0},
                {'path': 'IM_27.csv', 'target_hz': 100.01, 'base_hz': 100.0},
                {'path': 'IM_40.csv', 'target_hz': 100.01, 'base_hz': 100.0},
                {'path': 'IM_46.csv', 'target_hz': 100.01, 'base_hz': 100.0},
                {'path': 'IM_28.csv', 'target_hz': 100.005, 'base_hz': 100.0},
                {'path': 'IM_41.csv', 'target_hz': 100.005, 'base_hz': 100.0},
                {'path': 'IM_47.csv', 'target_hz': 100.005, 'base_hz': 100.0},
                {'path': 'IM_29.csv', 'target_hz': 200.0, 'base_hz': 200.0},
                {'path': 'IM_42.csv', 'target_hz': 200.0, 'base_hz': 200.0},
                {'path': 'IM_48.csv', 'target_hz': 200.0, 'base_hz': 200.0},
                {'path': 'IM_30.csv', 'target_hz': 200.01, 'base_hz': 200.0},
                {'path': 'IM_43.csv', 'target_hz': 200.01, 'base_hz': 200.0},
                {'path': 'IM_49.csv', 'target_hz': 200.01, 'base_hz': 200.0},
                {'path': 'IM_31.csv', 'target_hz': 200.005, 'base_hz': 200.0},
                {'path': 'IM_44.csv', 'target_hz': 200.005, 'base_hz': 200.0},
                {'path': 'IM_50.csv', 'target_hz': 200.005, 'base_hz': 200.0}]}

SCHEDULE_PRESETS = {}
RESOLUTION_FILES = [dict(item) for item in FILES["resolution"]]
_resolution_examples = {}
for item in RESOLUTION_FILES:
    if abs(item["target_hz"] - item["base_hz"] - 0.005) < 1e-8:
        _resolution_examples.setdefault(item["base_hz"], Path(item["path"]).stem)

# Full scale for new recordings; nominal g/count is derived automatically.
ACCEL_FULL_SCALE_G = 16.0
# Previously recorded +/-4 g files keep their original conversion.
# Remove/change an entry if that exact file has been replaced by a new recording.
ACCEL_FULL_SCALE_OVERRIDES = {
    "accel_1.csv": 4.0,
    "accel_2.csv": 4.0,
    "accel_3.csv": 4.0,
    "accel_4.csv": 4.0,
}
CONFIG = {
    'accel_conversion': {"full_scale_g": ACCEL_FULL_SCALE_G,
                         "file_full_scale_g": ACCEL_FULL_SCALE_OVERRIDES.copy(),
                         "unit": "g", "basis": "ASM330LHH nominal sensitivity by recording full scale",
                         "source": "https://www.st.com/resource/en/datasheet/asm330lhh.pdf"},
    'data_root': data_root('Data10'),
    'sensors': ["IM"],
    'sections': {'noise': False, 'sweep': False, 'resolution': True, 'volume': False},
    'fft': {"pad_factor": 8, "interpolation_scale": "log", "interpolation_points": 3,
            "max_gap_over_median": 3.0, "dt_cv_warning": 0.01,
            "gap_policy": "diagnostic"},
    'sine_fit': {
        "search_halfwidth_native_bins": 2.0, "coarse_points": 17,
        "frequency_tolerance_hz": 1e-9,
        "timestamp_basis": "recorded Time_us; relation to actual acquisition instants unconfirmed",
    },
    'resolution': {
        "files": RESOLUTION_FILES,
        "file_mapping_source": "User-supplied nominal mapping; explicit FILES list contains 27 recordings, three per setting",
        "repeat_grouping_basis": "User-grouped nominal settings; other acquisition conditions not independently verified",
        "desired_records_per_setting": 2,
        "start_s": 0.0,
        "windows_s": {"primary_100s": [0.0, 100.0], "trimmed_90s": [5.0, 95.0]},
    },
    'plots': {
        "relative_frequency_method": "global_padded_3point",
        "spectra_per_page": 8, "show_all_spectra": False,
        "show_strict_gap_tables": False,
        "spectrogram_segment_s": 2.0, "spectrogram_overlap": 0.75,
        "spectrogram_frequency_scales": ["linear", "log"],
        "spectrogram_nominal_marker": "o", "spectrogram_nominal_color": "cyan", "spectrogram_nominal_size": 18.0,
        "sweep_view": {"enabled": True, "min_hz": 10.0, "max_hz": None, "include_hz": None}, "spectrogram_frequency_limits_hz": [0, None],
        "spectrogram_dynamic_range_db": 100.0, "spectrogram_cmap": "viridis",
        "frequency_error_zoom_hz": [-0.15, 0.15],
        "interpolation_example_files": list(_resolution_examples.values()),
    },
}


In [ ]:
# sinewave-cell: inactive_section_defaults
# Inactive sections required by shared helpers; no other recordings are analyzed.
CONFIG.update({
    'frequency_range': {"min_hz": None, "max_hz": None, "include_hz": None},
    'sweep': {"IM": [], "accel": []},
    'noise': {"IM": [], "accel": []},
    'volume': {"IM": [], "accel": []},
    'sensitivity_trims_s': [],
    'snr': {"enabled": False, "sensors": [], "noise_files": {}, "band_halfwidth_hz": 1.0, "noise_segment_s": 6.0, "noise_overlap": 0.5},
    'target_diagnostics': {"enabled": True, "minimum_hz": 0.1, "search_halfwidth_hz": None},
    'volume_subwindow_s': 5.0,
    'volume_reference': {"conditions_confirmed": False, "pairs": [], "reference_level": 10, "amplitude_key": "global_padded_parabolic_amplitude", "plateau_fraction_threshold": 0.01},
})


## Check the schedule

Preview configured windows before reading the recordings. Filenames do not establish experimental conditions.


In [ ]:
# sinewave-cell: schedule_helpers
def _finite(value, name, minimum=0.0, positive=False):
    if isinstance(value, bool) or not isinstance(value, (int, float, np.number)):
        raise ValueError(f"{name} must be a number")
    value = float(value)
    if not np.isfinite(value) or value < minimum or (positive and value <= 0):
        raise ValueError(f"Invalid {name}: {value}")
    return value


def build_plan(config):
    """Build every acquisition interval BEFORE applying sweep-frequency selection."""
    c = deepcopy(config)
    root = Path(c["data_root"]).expanduser().resolve()
    if not c["sensors"] or len(set(c["sensors"])) != len(c["sensors"]) or set(c["sensors"]) - {"IM", "accel"}:
        raise ValueError("sensors must be a nonempty unique subset of ['IM', 'accel']")
    if set(c["sections"]) != {"noise", "sweep", "resolution", "volume"} or not all(isinstance(v, bool) for v in c["sections"].values()):
        raise ValueError("sections must contain four boolean switches")
    fft = c["fft"]
    if isinstance(fft["pad_factor"], bool) or not isinstance(fft["pad_factor"], int) or fft["pad_factor"] < 1:
        raise ValueError("fft.pad_factor must be a positive integer")
    if fft.get("gap_policy", "diagnostic") not in {"diagnostic", "strict"}:
        raise ValueError("gap_policy must be diagnostic or strict")
    if fft["interpolation_scale"] not in {"log", "linear"}:
        raise ValueError("interpolation_scale must be log or linear")
    points = fft.get("interpolation_points", 3)
    if isinstance(points, bool) or not isinstance(points, (int, np.integer)) or points not in (3, 5):
        raise ValueError("fft.interpolation_points must be 3 or 5")
    for key in ("max_gap_over_median", "dt_cv_warning"):
        _finite(fft[key], key, positive=True)
    for trim in c["sensitivity_trims_s"]:
        _finite(trim, "sensitivity trim")
    page_size = c["plots"]["spectra_per_page"]
    if isinstance(page_size, bool) or not isinstance(page_size, int) or not 1 <= page_size <= 16:
        raise ValueError("spectra_per_page must be an integer from 1 through 16")
    _finite(c["volume_subwindow_s"], "volume_subwindow_s", positive=True)
    target_config = c["target_diagnostics"]
    _finite(target_config["minimum_hz"], "target ranking minimum")
    if target_config["search_halfwidth_hz"] is not None:
        _finite(target_config["search_halfwidth_hz"], "target halfwidth", positive=True)
    plots = c["plots"]
    _finite(plots["spectrogram_segment_s"], "spectrogram duration", positive=True)
    overlap = _finite(plots["spectrogram_overlap"], "spectrogram overlap")
    if overlap >= 1:
        raise ValueError("spectrogram_overlap must be below 1")
    scales = plots["spectrogram_frequency_scales"]
    if not isinstance(scales, list) or not scales or len(set(scales)) != len(scales) or set(scales) - {"linear", "log"}:
        raise ValueError("spectrogram_frequency_scales needs unique linear/log entries")
    _finite(plots["spectrogram_nominal_size"], "spectrogram marker size", positive=True)
    view = plots["sweep_view"]
    if not isinstance(view["enabled"], bool):
        raise ValueError("sweep_view.enabled must be boolean")
    vlo = 0 if view["min_hz"] is None else _finite(view["min_hz"], "sweep_view.min_hz")
    vhi = np.inf if view["max_hz"] is None else _finite(view["max_hz"], "sweep_view.max_hz", positive=True)
    if vhi < vlo:
        raise ValueError("sweep_view.max_hz must not be smaller than min_hz")
    if view["include_hz"] is not None:
        for value in view["include_hz"]:
            _finite(value, "sweep_view.include_hz", positive=True)
    snr = c["snr"]
    if not isinstance(snr["enabled"], bool) or set(snr["sensors"]) - {"IM", "accel"}:
        raise ValueError("Invalid SNR switch/sensors")
    _finite(snr["band_halfwidth_hz"], "snr.band_halfwidth_hz", positive=True)
    _finite(snr["noise_segment_s"], "snr.noise_segment_s", positive=True)
    if not 0 <= _finite(snr["noise_overlap"], "snr.noise_overlap") < 1:
        raise ValueError("snr.noise_overlap must be below 1")
    for sensor in snr["sensors"]:
        labels = snr["noise_files"].get(sensor, [])
        if not labels or len(set(labels)) != len(labels):
            raise ValueError("Explicit unique noise files are required per SNR sensor")
    _finite(plots["spectrogram_dynamic_range_db"], "spectrogram dynamic range", positive=True)
    selection = c["frequency_range"]
    low, high = selection["min_hz"], selection["max_hz"]
    low = 0.0 if low is None else _finite(low, "min_hz")
    high = np.inf if high is None else _finite(high, "max_hz", positive=True)
    if high < low:
        raise ValueError("frequency_range.max_hz is smaller than min_hz")
    includes = selection["include_hz"]
    if includes is not None:
        includes = [_finite(v, "include_hz", positive=True) for v in includes]
    rows = []

    def add(sensor, role, entry, index, target, start, end, trim_start=0, trim_end=0, **extra):
        path = Path(entry["path"]).expanduser()
        path = (path if path.is_absolute() else root / path).resolve()
        start = _finite(start, "segment start")
        trim_start, trim_end = _finite(trim_start, "trim_start_s"), _finite(trim_end, "trim_end_s")
        if target is not None:
            target = _finite(target, "target_hz", positive=True)
        if end is not None:
            end = _finite(end, "segment end", positive=True)
            if end - trim_end <= start + trim_start:
                raise ValueError(f"Empty or over-trimmed interval: {path.name} {index}")
        selected = role != "sweep" or (low <= target <= high and (
            includes is None or any(np.isclose(target, value, rtol=0, atol=1e-10) for value in includes)))
        rows.append({"sensor": sensor, "role": role, "file_label": entry.get("label", path.stem),
                     "path": str(path), "segment_index": index, "target_hz": target,
                     "nominal_start_s": start, "nominal_end_s": end,
                     "analysis_start_s": start + trim_start,
                     "analysis_end_s": None if end is None else end - trim_end,
                     "trim_start_s": trim_start, "trim_end_s": trim_end,
                     "selected": selected, "window_variant": "primary", **extra})

    def bounds(entry, n):
        explicit = entry.get("segment_bounds_s")
        if explicit is None:
            start = _finite(entry["start_s"], "start_s")
            dwell = _finite(entry["dwell_s"], "dwell_s", positive=True)
            return [(start + i * dwell, start + (i + 1) * dwell) for i in range(n)]
        if len(explicit) != n:
            raise ValueError(f"{entry['path']}: need {n} explicit segment bounds")
        result = []
        for pair in explicit:
            if len(pair) != 2:
                raise ValueError("Each segment_bounds_s entry needs [start, end]")
            a, b = (_finite(v, "explicit bound") for v in pair)
            if b <= a or (result and a < result[-1][1]):
                raise ValueError("Explicit bounds must be ordered, non-overlapping, and nonempty")
            result.append((a, b))
        return result

    for sensor in c["sensors"]:
        if c["sections"]["noise"]:
            for entry in c["noise"][sensor]:
                add(sensor, "noise", entry, 1, None, entry["start_s"], entry["end_s"],
                    noise_condition=entry["condition"], timing_source="configured_noise_window")
        if c["sections"]["sweep"]:
            for entry in c["sweep"][sensor]:
                if not isinstance(entry.get("enabled", True), bool):
                    raise ValueError("sweep enabled must be boolean")
                if not entry.get("enabled", True):
                    continue
                freqs = entry.get("frequencies_hz")
                if freqs is None:
                    freqs = SCHEDULE_PRESETS.get(Path(entry["path"]).name)
                if freqs is None or len(freqs) == 0:
                    raise ValueError(f"Supply the complete frequencies_hz for {entry['path']}")
                for index, (target, (a, b)) in enumerate(zip(freqs, bounds(entry, len(freqs))), 1):
                    add(sensor, "sweep", entry, index, target, a, b,
                        entry["trim_start_s"], entry["trim_end_s"],
                        timing_source="explicit_bounds" if entry.get("segment_bounds_s") is not None else "nominal_schedule")
        if sensor == "IM" and c["sections"]["resolution"]:
            spec = c["resolution"]
            if not spec["windows_s"]:
                raise ValueError("resolution.windows_s must not be empty")
            start = _finite(spec["start_s"], "resolution start")
            for entry in spec["files"]:
                for index, (variant, pair) in enumerate(spec["windows_s"].items(), 1):
                    if len(pair) != 2:
                        raise ValueError("resolution window needs [start, end]")
                    a, b = (_finite(v, "resolution relative bound") for v in pair)
                    add(sensor, "resolution", entry, index, entry["target_hz"], start+a, start+b,
                        base_hz=_finite(entry["base_hz"], "base_hz", positive=True),
                        window_variant=variant, timing_source="configured_resolution_window")
        if c["sections"]["volume"]:
            for entry in c["volume"][sensor]:
                if not entry["levels"]:
                    raise ValueError("Volume levels must not be empty")
                for index, (level, (a, b)) in enumerate(zip(entry["levels"], bounds(entry, len(entry["levels"]))), 1):
                    add(sensor, "volume", entry, index, entry["target_hz"], a, b,
                        entry["trim_start_s"], entry["trim_end_s"], volume_setting=_finite(level, "volume"),
                        timing_source="explicit_bounds" if entry.get("segment_bounds_s") is not None else "nominal_schedule")
    plan = pd.DataFrame(rows)
    if plan.empty or not plan["selected"].any():
        raise ValueError("Configuration selects no analysis windows")
    keys = ["sensor", "role", "path", "segment_index", "window_variant"]
    if plan.duplicated(keys).any():
        raise ValueError("Duplicate file/segment configuration; do not count it as another repeat")
    return plan



def select_sweep_view(table, settings):
    """A display-only selection; never reconstruct or shift acquisition windows."""
    if table.empty:
        return table.copy()
    f = pd.to_numeric(table["target_hz"], errors="coerce")
    lo = 0.0 if settings["min_hz"] is None else float(settings["min_hz"])
    hi = np.inf if settings["max_hz"] is None else float(settings["max_hz"])
    mask = f.between(lo, hi)
    if settings["include_hz"] is not None:
        chosen = np.asarray(settings["include_hz"], float)
        mask &= np.any(np.isclose(f.to_numpy()[:, None], chosen[None, :], rtol=0, atol=1e-10), axis=1)
    return table.loc[mask].copy()


def sweep_view_label(settings):
    lo = "lowest" if settings["min_hz"] is None else f'{settings["min_hz"]:g} Hz'
    hi = "highest selected" if settings["max_hz"] is None else f'{settings["max_hz"]:g} Hz'
    text = f"selected view: {lo} to {hi}"
    if settings["include_hz"] is not None:
        text += " (explicit frequency list)"
    return text


In [ ]:
# sinewave-cell: schedule_preview
PLANNED_SEGMENTS = build_plan(CONFIG)
display(PLANNED_SEGMENTS[["sensor", "role", "file_label", "segment_index", "target_hz",
                          "analysis_start_s", "analysis_end_s", "window_variant", "selected"]])
display(PLANNED_SEGMENTS.groupby(["sensor", "role"], sort=False)["selected"].agg(["size", "sum"]))
print("size = full schedule; sum = selected windows. Frequency selection never shifts timestamps.")


## Methods and quality checks

`Time_us` is converted to seconds relative to the first timestamp. IM output remains
`Data28` counts; accelerometer `Raw_Z` uses the configured nominal g/count conversion
(0.000488 at +/-16 g; 0.000122 at +/-4 g). This is not individual sensor calibration.
Original CSV files are read only. Missing, nonnumeric, duplicate or decreasing
timestamps are rejected without sorting or dropping samples.

Within each half-open analysis window, the FFT uses `fs=(N-1)/(t_last-t_first)` and
linear interpolation onto N evenly spaced times. After mean removal and a periodic
Hann window, one-sided peak amplitude is `2*abs(rFFT)/sum(window)` (without doubling
DC or Nyquist). Native and 8x zero-padded global positive-frequency peaks are kept
separately; three-point log-amplitude interpolation refines each selected peak.
Invalid or tied peaks retain explicit missing estimates; no alternative peak is
substituted. Raw demeaned RMS and peak-to-peak amplitude remain separate metrics.

The existing diagnostic gap policy retains computed estimates with timing warnings;
strict results retain the 3x median-gap check. `fft_computed` and `fft_valid` have
different meanings. Actual windows, jitter, gaps, samples per cycle, native/padded
bin spacing and Hann ENBW are saved. No additional filter is applied. Sparse
sampling and interpolation can bias amplitudes. Zero padding does not improve
physical resolving power or calibrate timestamps.

The primary estimator is `global_padded_3point`: 8x padding and a three-point log
fit at the global peak. Native/padded maximum bins, three/five-point fits and an
original-timestamp sine fit remain separate comparisons. Five-point fits are not
assumed better; nominal settings do not select the winning estimator.

The original-time fit uses all retained raw samples in an equal-weight fit of
`c+a*cos(2*pi*f*t)+b*sin(2*pi*f*t)`, without resampling, filtering or Hann weighting.
It searches around the observed native FFT maximum, within two native bins.
The assumption that Time_us represents acquisition instants remains unverified;
fit residual RMS is not a measured noise floor or SNR.

For each sensor/base/window/method, relative frequency is
`1000*(estimate_hz-baseline_mean_hz)` in mHz. The baseline contains only valid
records at the base setting; a missing baseline remains missing. Bars show
across-record sample SD, not the uncertainty of a baseline difference. The
uncorrected absolute-error plot uses `(estimate-target)/target*1e6` ppm.

Each of the nine settings has three selected recordings (27 files). The 100 s
main and 90 s trimmed windows are separate analyses of the same records, not
additional repeats. Historical checks found timing warnings in all 54 windows;
the diagnostic and strict results remain separate. Relative display is not a
frequency calibration, and single-tone repeatability is not simultaneous
two-tone resolution.


In [ ]:
# sinewave-cell: fft_core
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.signal.windows import hann


def audit_timestamps(t, y):
    """Audit without sorting, dropping, merging, or repairing original samples."""
    t, y = np.asarray(t, dtype=float), np.asarray(y, dtype=float)
    if t.ndim != 1 or y.ndim != 1 or t.size != y.size:
        raise ValueError('Time and signal must be equally sized one-dimensional arrays.')
    if t.size < 2:
        raise ValueError('At least two recorded samples are required.')
    if not np.all(np.isfinite(t)) or not np.all(np.isfinite(y)):
        raise ValueError('Nonfinite timestamps or signal values: no rows were dropped.')
    dt = np.diff(t)
    if np.any(dt <= 0):
        raise ValueError(f'Timestamp integrity failure: {(dt == 0).sum()} duplicate and '
                         f'{(dt < 0).sum()} decreasing intervals; samples were not reordered.')
    median = float(np.median(dt))
    mean = float(np.mean(dt))
    return dict(sample_count=int(t.size), recorded_start_s=float(t[0]),
                recorded_end_s=float(t[-1]), recorded_span_s=float(t[-1] - t[0]),
                dt_mean_s=mean, dt_median_s=median, dt_min_s=float(dt.min()),
                dt_max_s=float(dt.max()), dt_std_s=float(dt.std(ddof=0)),
                dt_cv=float(dt.std(ddof=0) / mean),
                max_gap_over_median=float(dt.max() / median),
                intervals_over_1p5_median=int(np.sum(dt > 1.5 * median)),
                timestamp_duplicates=0, timestamp_reversals=0, nonfinite_samples=0,
                samples_removed=0)


def accel_conversion_for(path, settings):
    nominal_g_per_count = {2.0: 0.000061, 4.0: 0.000122, 8.0: 0.000244, 16.0: 0.000488}
    if settings is None or settings.get('unit', 'g') != 'g':
        raise ValueError('Supply accel_conversion with full_scale_g (2, 4, 8 or 16) and unit g.')
    overrides = settings.get('file_full_scale_g', {})
    path = Path(path).expanduser().resolve()
    full_scale = overrides.get(path.name, settings.get('full_scale_g'))
    # Explicit full paths take precedence over basename overrides.
    for filename, value in overrides.items():
        candidate = Path(filename).expanduser()
        if candidate.is_absolute() and candidate.resolve() == path:
            full_scale = value
    if isinstance(full_scale, bool) or full_scale not in nominal_g_per_count:
        raise ValueError(f'{path.name}: full_scale_g must be 2, 4, 8 or 16; got {full_scale!r}.')
    return float(full_scale), nominal_g_per_count[full_scale]


def load_record(path, sensor, accel_conversion=None):
    """Read Time_us plus Data28 (IM) or Raw_Z (accel); convert accel counts to g using the selected per-file full scale."""
    aliases = {'im': 'IM', 'inductive microphone': 'IM',
               'accel': 'accel', 'accelerometer': 'accel'}
    key = str(sensor).strip().lower()
    if key not in aliases:
        raise ValueError('sensor must be IM or accel; units/conversions are not inferred.')
    sensor = aliases[key]
    column = 'Data28' if sensor == 'IM' else 'Raw_Z'
    path = Path(path).expanduser().resolve(strict=True)
    frame = pd.read_csv(path)
    if 'Time_us' not in frame or column not in frame:
        raise ValueError(f'{path.name}: required columns Time_us and {column}. '
                         f'Observed columns: {list(frame.columns)}')
    # Strict conversion: malformed entries and NaNs fail instead of being dropped.
    timestamps_us = pd.to_numeric(frame['Time_us'], errors='raise').to_numpy(dtype=float)
    y = pd.to_numeric(frame[column], errors='raise').to_numpy(dtype=float)
    if len(timestamps_us) == 0:
        raise ValueError(f'{path.name}: empty record.')
    t = (timestamps_us - timestamps_us[0]) * 1e-6
    audit = audit_timestamps(t, y)
    full_scale, scale = (accel_conversion_for(path, accel_conversion)
                         if sensor == 'accel' else (None, 1.0))
    unit = 'g' if sensor == 'accel' else 'Data28 counts'
    raw_counts = y.copy()
    y = y * scale
    return dict(t=t, y=y, raw_counts=raw_counts, path=str(path), sensor=sensor, unit=unit,
                metadata=dict(input_path=str(path), sensor=sensor, signal_column=column,
                              time_column='Time_us', timestamp_origin_us=float(timestamps_us[0]),
                              timestamp_to_seconds=1e-6, signal_scale=scale, input_signal_unit='counts',
                              nominal_full_scale_g=full_scale,
                              signal_unit=unit, timestamp_audit=audit))


def _fft_arrays(y, fs, nfft):
    """Periodic Hann, mean removal, coherent-gain peak amplitude, PSD density."""
    n = len(y)
    window = hann(n, sym=False)
    transform = np.fft.rfft((y - np.mean(y)) * window, n=int(nfft))
    f = np.fft.rfftfreq(int(nfft), d=1.0 / fs)
    amplitude = np.abs(transform) / window.sum()
    psd = np.abs(transform)**2 / (fs * np.sum(window**2))
    # DC and (only for an even transform length) the exact Nyquist bin occur once.
    interior = slice(1, -1) if int(nfft) % 2 == 0 else slice(1, None)
    amplitude[interior] *= 2.0
    psd[interior] *= 2.0
    enbw = fs * np.sum(window**2) / window.sum()**2
    return f, amplitude, psd, float(enbw)


def _quadratic_peak_fits(f, amplitude, indices, interpolation_scale='log', interpolation_points=3):
    """Fit actual neighboring bins in bulk; invalid fits stay missing, with no fallback.

    Coefficients are in centered bin units, highest power first. Three points
    retain the original closed form; five points use unweighted quadratic OLS.
    """
    if interpolation_scale not in ('log', 'linear'):
        raise ValueError('interpolation_scale must be log or linear.')
    if (isinstance(interpolation_points, bool)
            or not isinstance(interpolation_points, (int, np.integer))
            or interpolation_points not in (3, 5)):
        raise ValueError('interpolation_points must be 3 or 5.')
    f, amplitude = np.asarray(f, dtype=float), np.asarray(amplitude, dtype=float)
    indices = np.asarray(indices, dtype=int)
    if f.ndim != 1 or f.shape != amplitude.shape or indices.ndim != 1:
        raise ValueError('Expected matching one-dimensional FFT arrays and peak indices.')
    n, radius = len(indices), interpolation_points // 2
    result = dict(parabolic_hz=np.full(n, np.nan), parabolic_amplitude=np.full(n, np.nan),
                  interpolation_status=np.full(n, 'unavailable', dtype=object),
                  coefficients=np.full((n, 3), np.nan), delta=np.full(n, np.nan))
    status = result['interpolation_status']
    left_missing, right_missing = indices < radius, indices + radius >= len(f)
    status[left_missing] = 'insufficient_left_neighbors'
    status[right_missing & ~left_missing] = ('no_right_neighbor' if radius == 1
                                            else 'insufficient_right_neighbors')
    active = np.flatnonzero(~(left_missing | right_missing))
    if not len(active):
        return result
    offsets = np.arange(-radius, radius + 1, dtype=float)
    samples = amplitude[indices[active, None] + offsets.astype(int)]
    good = np.all(np.isfinite(samples) & (samples > 0), axis=1)
    status[active[~good]] = 'zero_or_invalid_neighbor'
    active, samples = active[good], samples[good]
    good = ((samples[:, radius] > samples[:, radius - 1])
            & (samples[:, radius] > samples[:, radius + 1]))
    status[active[~good]] = 'not_unique_local_maximum'
    active, samples = active[good], samples[good]
    if not len(active):
        return result
    curve = np.log(samples) if interpolation_scale == 'log' else samples
    with np.errstate(over='ignore', invalid='ignore', divide='ignore'):
        if interpolation_points == 3:
            left, center, right = curve.T
            coefficients = np.column_stack(((left - 2 * center + right) / 2,
                                            (right - left) / 2, center))
        else:
            design = np.column_stack((offsets**2, offsets, np.ones_like(offsets)))
            # Center the ordinate for numerical stability; restore the intercept.
            coefficients = (curve - curve[:, [radius]]) @ np.linalg.pinv(design).T
            coefficients[:, 2] += curve[:, radius]
        good = np.all(np.isfinite(coefficients), axis=1) & (coefficients[:, 0] < 0)
        status[active[~good]] = 'nonconcave_or_invalid_curve'
        active, coefficients = active[good], coefficients[good]
        a, b, c = coefficients.T
        delta = -b / (2 * a)
        good = np.isfinite(delta) & (abs(delta) <= 0.5 + 1e-12)
        status[active[~good]] = 'vertex_outside_adjacent_half_bins'
        active, coefficients, delta = active[good], coefficients[good], delta[good]
        a, b, c = coefficients.T
        vertex = c + b * delta + a * delta**2
        peaks = np.exp(vertex) if interpolation_scale == 'log' else vertex
        k = indices[active]
        df = f[k + 1] - f[k]
        frequency = f[k] + delta * df
        good = (np.isfinite(peaks) & (peaks > 0) & np.isfinite(frequency)
                & (frequency > 0) & np.isfinite(df) & (df > 0))
    status[active[~good]] = 'invalid_vertex'
    active = active[good]
    status[active] = 'ok'
    result['parabolic_hz'][active] = frequency[good]
    result['parabolic_amplitude'][active] = peaks[good]
    result['coefficients'][active] = coefficients[good]
    result['delta'][active] = delta[good]
    return result


def _global_peak(f, amplitude, interpolation_scale='log', interpolation_points=3):
    """All positive rFFT bins; target frequency never enters this function."""
    result = dict(bin_hz=np.nan, bin_amplitude=np.nan, parabolic_hz=np.nan,
                  parabolic_amplitude=np.nan, interpolation_status='unavailable',
                  peak_tie_count=0, peak_exact_tie_count=0, peak_tie_relative_tolerance=1e-12,
                  peak_index=None, interpolation_points=interpolation_points)
    if len(f) < 2 or len(f) != len(amplitude):
        result['interpolation_status'] = 'insufficient_fft_bins'
        return result
    values = np.asarray(amplitude[1:])
    if not np.all(np.isfinite(values)) or values.max() <= 0:
        result['interpolation_status'] = 'zero_or_invalid_spectrum'
        return result
    peak = float(values.max())
    ties = np.flatnonzero(np.isclose(values, peak, rtol=1e-12, atol=0.0)) + 1
    # Raw-bin estimate is the actual strict argmax, even for near ties.
    # np.argmax chooses the lower index only when maxima are EXACTLY equal.
    k = int(np.argmax(values)) + 1
    exact_count = int(np.count_nonzero(values == peak))
    result.update(bin_amplitude=peak, bin_hz=float(f[k]), peak_index=k,
                  peak_tie_count=int(len(ties)), peak_exact_tie_count=exact_count)
    if len(ties) != 1:
        result['interpolation_status'] = ('tied_global_maximum' if exact_count > 1
                                          else 'near_tied_global_maximum')
        return result
    fitted = _quadratic_peak_fits(f, amplitude, [k], interpolation_scale, interpolation_points)
    result.update(parabolic_hz=float(fitted['parabolic_hz'][0]),
                  parabolic_amplitude=float(fitted['parabolic_amplitude'][0]),
                  interpolation_status=str(fitted['interpolation_status'][0]))
    return result



def analyze_segment(record, start_s, end_s, target_hz, fft_config=None):
    """Analyze an explicitly supplied half-open window; never shorten unsupported windows.

    raw_rms is the RMS of the ORIGINAL retained samples after removing their mean.
    raw_rms_including_dc is separately retained. All amplitudes are in input units.
    Nonuniform samples are linearly interpolated to N points spanning first-to-last
    retained time; fs=(N-1)/span, native_df=fs/N. Padding changes only the FFT grid.
    Large internal gaps retain strict eligibility=False; diagnostic mode computes
    explicitly assumption-bearing FFTs without changing the threshold. Boundary
    coverage failures remain unavailable in both modes; raw metrics are retained.
    nominal_cycles = target*N/fs (effective FFT duration); requested_cycles uses the
    requested window duration. Fixed raw excursion warnings: crest >10, robust z >20.
    """
    cfg = dict(pad_factor=8, interpolation_scale='log', interpolation_points=3, max_gap_over_median=3.0,
               dt_cv_warning=0.01, gap_policy="diagnostic")
    if fft_config:
        unknown = set(fft_config) - set(cfg)
        if unknown:
            raise ValueError(f'Unknown FFT settings: {sorted(unknown)}')
        cfg.update(fft_config)
    pad = cfg['pad_factor']
    if isinstance(pad, bool) or not np.isfinite(pad) or int(pad) != pad or pad < 1:
        raise ValueError('pad_factor must be a positive integer.')
    if cfg['gap_policy'] not in ('diagnostic', 'strict'):
        raise ValueError('gap_policy must be diagnostic or strict.')
    if cfg['interpolation_scale'] not in ('log', 'linear'):
        raise ValueError('interpolation_scale must be log or linear.')
    points = cfg['interpolation_points']
    if isinstance(points, bool) or not isinstance(points, (int, np.integer)) or points not in (3, 5):
        raise ValueError('interpolation_points must be 3 or 5.')
    if (not np.isfinite(cfg['max_gap_over_median']) or not np.isfinite(cfg['dt_cv_warning'])
            or cfg['max_gap_over_median'] < 1 or cfg['dt_cv_warning'] < 0):
        raise ValueError('Invalid timestamp quality thresholds.')
    t = np.asarray(record['t'], dtype=float)
    y = np.asarray(record['y'], dtype=float)
    full_audit = audit_timestamps(t, y)
    start_s, end_s = float(start_s), float(end_s)
    if not np.isfinite(start_s) or not np.isfinite(end_s) or end_s <= start_s:
        raise ValueError('Window must have finite start < end.')
    # A conventional [0, N/fs) capture ends one interval after the last sample.
    # This is the only permitted nominal end extension; it never adds a sample.
    nominal_end_tolerance = full_audit['dt_mean_s']
    supported_end = float(t[-1] + nominal_end_tolerance)
    floating_tolerance = max(1e-10, np.finfo(float).eps * max(1.0, abs(end_s)) * 16)
    if start_s < t[0] - floating_tolerance or end_s > supported_end + floating_tolerance:
        raise ValueError(f'Unsupported window [{start_s:g}, {end_s:g}) s. '
                         f'Recorded support starts at {t[0]:g} s and nominal '
                         f'half-open support ends at {supported_end:g} s; not shortened.')
    mask = (t >= start_s) & (t < end_s)
    retained_t, retained_y = t[mask], y[mask]
    if len(retained_t) < 4:
        raise ValueError('Fewer than four samples in retained window; FFT unsupported.')
    audit = audit_timestamps(retained_t, retained_y)
    n = len(retained_t)
    fs = (n - 1) / (retained_t[-1] - retained_t[0])
    target = np.nan if target_hz is None else float(target_hz)
    if not np.isnan(target) and (not np.isfinite(target) or target <= 0):
        raise ValueError('target_hz must be positive, None, or NaN.')
    centered = retained_y - retained_y.mean()
    raw_rms = float(np.sqrt(np.mean(centered**2)))
    crest = float(np.max(np.abs(centered)) / raw_rms) if raw_rms > 0 else 0.0
    deviations = np.abs(retained_y - np.median(retained_y))
    robust_scale = float(1.4826 * np.median(deviations))
    robust_z = (float(deviations.max() / robust_scale) if robust_scale > 0
                else (float('inf') if deviations.max() > 0 else 0.0))
    start_boundary_gap = max(0.0, float(retained_t[0] - start_s))
    end_boundary_gap = max(0.0, float(end_s - retained_t[-1]))
    boundary_reference_dt = full_audit['dt_median_s']
    intervals = np.diff(retained_t)
    gap_cutoff_ratio = cfg['max_gap_over_median'] + max(1e-9, abs(cfg['max_gap_over_median']) * 1e-9)
    gap_intervals = intervals / audit['dt_median_s'] > gap_cutoff_ratio
    raw_min, raw_max = float(retained_y.min()), float(retained_y.max())
    row = dict(sample_count=n, actual_start_s=float(retained_t[0]),
               actual_end_s=float(retained_t[-1]), analysis_start_s=start_s,
               analysis_end_s=end_s, requested_duration_s=end_s - start_s,
               effective_fft_duration_s=float(n / fs),
               start_to_first_sample_s=start_boundary_gap,
               last_sample_to_end_s=end_boundary_gap,
               boundary_gap_reference_dt_s=boundary_reference_dt,
               start_boundary_gap_over_median=start_boundary_gap / boundary_reference_dt,
               end_boundary_gap_over_median=end_boundary_gap / boundary_reference_dt,
               retained_span_s=float(retained_t[-1] - retained_t[0]),
               nominal_end_tolerance_s=nominal_end_tolerance,
               end_after_last_recorded_sample_s=max(0.0, end_s - float(t[-1])),
               nominal_record_support_end_s=supported_end,
               end_uses_nominal_interval_tolerance=bool(end_s > t[-1]),
               floating_support_tolerance_s=floating_tolerance,
               fs_hz=float(fs), nyquist_hz=float(fs / 2), native_df_hz=float(fs / n),
               padded_df_hz=float(fs / (n * int(pad))), fft_length=n,
               padded_fft_length=n * int(pad), enbw_hz=np.nan,
               raw_mean=float(retained_y.mean()),
               raw_min_value=raw_min, raw_max_value=raw_max,
               raw_fraction_at_min=float(np.mean(retained_y == raw_min)),
               raw_fraction_at_max=float(np.mean(retained_y == raw_max)),
               raw_extrema_occupancy_fraction=float(np.mean((retained_y == raw_min) | (retained_y == raw_max))),
               gap_policy=cfg['gap_policy'], gap_comparison_ratio_tolerance=max(1e-9, abs(cfg['max_gap_over_median']) * 1e-9),
               gap_interval_count=int(gap_intervals.sum()),
               gap_interval_fraction=float(gap_intervals.mean()),
               gap_duration_sum_s=float(intervals[gap_intervals].sum()),
               gap_duration_fraction=float(intervals[gap_intervals].sum() / (retained_t[-1] - retained_t[0])),
               gap_fraction_definition='duration of intervals above threshold / retained timestamp span; not a measured lost-sample fraction',
               analysis_status='fft_unavailable', fft_computed=False,
               strict_gap_check_passed=False, window_coverage_passed=False,
               strict_rejection_flags='',

               raw_rms=raw_rms, raw_crest_factor=crest, raw_max_robust_z=robust_z,
               raw_robust_scale=robust_scale, raw_crest_factor_warning_threshold=10.0,
               raw_max_robust_z_warning_threshold=20.0,
               raw_rms_including_dc=float(np.sqrt(np.mean(retained_y**2))),
               raw_peak_to_peak=float(np.ptp(retained_y)), target_hz=target,
               nominal_cycles=float(target * n / fs),
               requested_cycles=float(target * (end_s - start_s)),
               samples_per_target_cycle=float(fs / target),
               max_gap_target_cycles=float(audit['dt_max_s'] * target),
               target_within_nyquist=bool(np.isfinite(target) and target < fs / 2),
               pad_factor=int(pad), interpolation_scale=cfg['interpolation_scale'],
               interpolation_points=int(points), interpolation_polynomial_degree=2,
               interpolation_method='unweighted quadratic least squares in centered bin offsets',
               global_search='all strictly positive rFFT bins; DC excluded; strict np.argmax',
               global_peak_tie_relative_tolerance=1e-12,
               global_peak_tie_absolute_tolerance=0.0,
               global_peak_tie_policy='raw argmax retained; exact/near ties reject interpolation',
               timestamp_resampling='linear; N samples spanning retained first-to-last time',
               preprocessing='remove mean; periodic Hann; no bandpass or other filter',
               amplitude_definition='one-sided sinusoidal peak amplitude; divide by sum(window)',
               raw_rms_definition='sqrt(mean((original retained y - mean(y))**2))',
               psd_definition='one-sided abs(FFT)**2/(fs*sum(window**2))',
               quality_flags='', fft_valid=False)
    row.update(audit)
    estimators = ('global_bin', 'global_parabolic', 'global_padded', 'global_padded_parabolic')
    for estimator in estimators:
        row[estimator + '_hz'] = np.nan
        row[estimator + '_amplitude'] = np.nan
        row[estimator + '_error_hz'] = np.nan
        for suffix in ('_hz', '_amplitude', '_error_hz'):
            row['strict_' + estimator + suffix] = np.nan
    row.update(raw_order_parabolic_hz=np.nan, raw_order_parabolic_error_hz=np.nan,
               raw_order_minus_resampled_hz=np.nan,
               global_interpolation_status='fft_unavailable',
               global_padded_interpolation_status='fft_unavailable',
               raw_order_interpolation_status='fft_unavailable',
               global_peak_tie_count=0, global_padded_peak_tie_count=0,
               global_peak_exact_tie_count=0, global_padded_peak_exact_tie_count=0)
    empty = np.array([], dtype=float)
    spectrum = dict(f=empty.copy(), amplitude=empty.copy(), psd=empty.copy(),
                    padded_f=empty.copy(), padded_amplitude=empty.copy(),
                    t=retained_t, y=retained_y, uniform_t=empty.copy(), uniform_y=empty.copy())
    flags = []
    if n < 8:
        flags.append('fewer_than_8_retained_samples')
    if crest > 10.0:
        flags.append('large_excursion_crest_factor')
    if robust_z > 20.0:
        flags.append('large_excursion_robust_z')
    if audit['dt_cv'] > cfg['dt_cv_warning']:
        flags.append('timestamp_jitter_linear_interpolation_may_bias_spectrum')
    if np.isfinite(target):
        if target >= fs / 2:
            flags.append('target_at_or_above_nyquist_no_alias_frequency_inferred')
        if target >= 0.9 * fs / 2:
            flags.append('target_at_or_above_90_percent_nyquist')
        if row['samples_per_target_cycle'] < 4:
            flags.append('fewer_than_4_samples_per_target_cycle')
        elif row['samples_per_target_cycle'] < 10:
            flags.append('between_4_and_10_samples_per_target_cycle')
        if row['nominal_cycles'] < 5:
            flags.append('fewer_than_5_target_cycles')
        if row['max_gap_target_cycles'] >= 0.5:
            flags.append('largest_timestamp_gap_at_least_half_target_cycle')
    internal_gap_warning = bool(gap_intervals.any())
    boundary_warnings = [boundary for boundary in ('start', 'end')
                         if row[boundary + '_boundary_gap_over_median'] > gap_cutoff_ratio]
    strict_flags = (['fft_rejected_large_timestamp_gap'] if internal_gap_warning else [])
    strict_flags.extend('fft_rejected_' + boundary + '_boundary_gap' for boundary in boundary_warnings)
    row['strict_rejection_flags'] = ';'.join(strict_flags)
    row['strict_gap_check_passed'] = not strict_flags
    row['window_coverage_passed'] = not boundary_warnings
    if boundary_warnings or (internal_gap_warning and cfg['gap_policy'] == 'strict'):
        flags.extend(strict_flags)
        row['quality_flags'] = ';'.join(flags)
        row['analysis_status'] = 'unsupported_boundary_coverage' if boundary_warnings else 'strict_gap_rejected'
        return row, spectrum
    if internal_gap_warning:
        flags.append('large_timestamp_gap_interpolated_diagnostic')
    uniform_t = np.linspace(retained_t[0], retained_t[-1], n)
    uniform_y = np.interp(uniform_t, retained_t, retained_y)
    spectrum.update(uniform_t=uniform_t, uniform_y=uniform_y)
    f, amplitude, psd, enbw = _fft_arrays(uniform_y, fs, n)
    pf, pa, _, _ = _fft_arrays(uniform_y, fs, n * int(pad))
    spectrum.update(f=f, amplitude=amplitude, psd=psd, padded_f=pf, padded_amplitude=pa)
    row['enbw_hz'] = enbw
    numerical_floor = 8 * np.finfo(float).eps * float(np.max(np.abs(retained_y)))
    if row['raw_rms'] <= numerical_floor:
        flags.append('constant_or_numerically_zero_ac_signal')
        row.update(global_interpolation_status='zero_signal',
                   global_padded_interpolation_status='zero_signal',
                   raw_order_interpolation_status='zero_signal',
                   quality_flags=';'.join(flags))
        return row, spectrum
    native = _global_peak(f, amplitude, cfg['interpolation_scale'], points)
    padded = _global_peak(pf, pa, cfg['interpolation_scale'], points)
    rf, ra, _, _ = _fft_arrays(retained_y, fs, n)
    raw_order = _global_peak(rf, ra, cfg['interpolation_scale'], points)
    # Same fs for raw-order diagnostic, which deliberately ignores timestamp jitter.
    for prefix, result in (('global', native), ('global_padded', padded)):
        bin_name = 'global_bin' if prefix == 'global' else 'global_padded'
        row[bin_name + '_hz'] = result['bin_hz']
        row[bin_name + '_amplitude'] = result['bin_amplitude']
        row[prefix + '_parabolic_hz'] = result['parabolic_hz']
        row[prefix + '_parabolic_amplitude'] = result['parabolic_amplitude']
        row[prefix + '_interpolation_status'] = result['interpolation_status']
        row[prefix + '_peak_tie_count'] = result['peak_tie_count']
        row[prefix + '_peak_exact_tie_count'] = result['peak_exact_tie_count']
        if result['interpolation_status'] != 'ok':
            flags.append(prefix + '_' + result['interpolation_status'])
    row['raw_order_parabolic_hz'] = raw_order['parabolic_hz']
    row['raw_order_interpolation_status'] = raw_order['interpolation_status']
    row['raw_order_minus_resampled_hz'] = raw_order['parabolic_hz'] - native['parabolic_hz']
    row['raw_order_parabolic_error_hz'] = raw_order['parabolic_hz'] - target
    for estimator in estimators:
        row[estimator + '_error_hz'] = row[estimator + '_hz'] - target
    if np.isfinite(target) and np.isfinite(native['bin_hz']):
        if abs(native['bin_hz'] - target) > max(2 * row['native_df_hz'], 0.05 * target):
            flags.append('global_peak_far_from_nominal_target')
    if abs(row['raw_order_minus_resampled_hz']) > 0.1 * row['native_df_hz']:
        flags.append('raw_order_and_resampled_estimates_differ')
    row['fft_computed'] = True
    row['fft_valid'] = row['strict_gap_check_passed']
    row['analysis_status'] = ('computed_with_timing_warning' if internal_gap_warning else 'computed')
    if row['strict_gap_check_passed']:
        for estimator in estimators:
            for suffix in ('_hz', '_amplitude', '_error_hz'):
                row['strict_' + estimator + suffix] = row[estimator + suffix]
    row['quality_flags'] = ';'.join(flags)
    return row, spectrum


In [ ]:
# sinewave-cell: plot_helpers
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal

# Plotting only: estimators and quality decisions come from the analysis tables.
_PLOT_METHODS = (
    ('global_bin_hz', 'Native FFT maximum', 'o'),
    ('global_parabolic_hz', 'Quadratic fit', 's'),
    ('global_padded_hz', 'Zero-padded FFT maximum', '^'),
    ('global_padded_parabolic_hz', 'Padded + quadratic fit', 'D'),
)
_PLOT_AMPLITUDES = (
    ('global_bin_amplitude', 'Native FFT peak amplitude', 'o'),
    ('global_parabolic_amplitude', 'Interpolated peak amplitude', 's'),
    ('global_padded_amplitude', 'Zero-padded peak amplitude', '^'),
    ('global_padded_parabolic_amplitude', 'Padded + quadratic fit peak amplitude', 'D'),
)


def _plot_style(ax, xlabel=None, ylabel=None, title=None):
    ax.grid(True, which='major', alpha=0.25)
    ax.grid(True, which='minor', alpha=0.09)
    if xlabel:
        ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title, loc='left', fontsize=11)
    ax.spines[['top', 'right']].set_visible(False)


def _plot_unit(frame_or_row):
    value = frame_or_row.get('unit', 'recorded units')
    if isinstance(value, pd.Series):
        values = value.dropna().astype(str).unique()
        return values[0] if len(values) == 1 else 'recorded units (see table)'
    return str(value)


def _plot_values(table, key):
    return pd.to_numeric(table.get(key, pd.Series(np.nan, index=table.index)), errors='coerce').to_numpy(dtype=float)


def _plot_legend(ax):
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        unique = dict(zip(labels, handles))
        ax.legend(unique.values(), unique.keys(), fontsize=8, frameon=False, loc='best')


def _plot_timing_mask(table):
    computed = table.get('fft_computed', table.get('fft_valid', pd.Series(False, index=table.index)))
    strict = table.get('fft_valid', pd.Series(False, index=table.index))
    return computed.fillna(False).astype(bool).to_numpy() & ~strict.fillna(False).astype(bool).to_numpy()


def _plot_timing_row(row):
    return bool(row.get('fft_computed', row.get('fft_valid', False))) and not bool(row.get('fft_valid', False))


def _plot_warning_points(ax, x, y, mask):
    good = np.asarray(mask) & np.isfinite(x) & np.isfinite(y)
    if good.any():
        ax.scatter(np.asarray(x)[good], np.asarray(y)[good], s=82, marker='o',
                   facecolors='none', edgecolors='tab:red', linewidths=1.1, zorder=7,
                   label='Timing warning: diagnostic estimate')


def _plot_warning_note(fig, table):
    n = int(_plot_timing_mask(table).sum())
    if n:
        fig.supxlabel(f'Open red circles: {n}/{len(table)} windows computed across timestamp gaps; '
                      'strict FFT eligibility failed. Interpret diagnostic estimates with timing uncertainty.',
                      fontsize=9, color='firebrick')


def _plot_spectral_envelope(frequency, values, maximum_points=6000):
    """Display only: maximum in each contiguous log-index bucket, exact peak retained.

    No estimator reads this envelope. All positive bins enter the estimators upstream.
    """
    f, v = np.asarray(frequency), np.asarray(values)
    if len(f) <= maximum_points:
        return f, v
    edges = np.unique(np.r_[0, np.geomspace(1, len(f), maximum_points).astype(int)])
    chosen = np.array([a + int(np.argmax(v[a:b])) for a, b in zip(edges[:-1], edges[1:])])
    chosen = np.unique(np.r_[0, chosen, len(f) - 1])
    return f[chosen], v[chosen]


def _plot_title(row):
    target = row.get('target_hz', np.nan)
    target_text = f'{float(target):g} Hz' if pd.notna(target) else 'noise'
    label = row.get('file_label', row.get('file_id', row.get('record_id', row.get('label', row.get('role', 'record')))))
    details = [str(label)]
    variant = row.get('window_variant', None)
    if pd.notna(variant) and str(variant):
        details.append(str(variant))
    segment = row.get('segment_index', row.get('segment_idx', row.get('segment', None)))
    if pd.notna(segment):
        details.append(f'segment {segment}')
    details.append(f'target: {target_text}')
    return ' | '.join(details)


def _plot_empty(title):
    fig, ax = plt.subplots(figsize=(14, 5), layout='constrained')
    ax.text(0.5, 0.5, 'No supported rows available for this plot.', ha='center', va='center', transform=ax.transAxes)
    ax.set_axis_off()
    fig.suptitle(title)
    return fig






def plot_spectra(items, title, per_page=8):
    """Full positive-frequency band; no display crop is used for peak estimation."""
    figures = []
    per_page = max(1, int(per_page))
    for first in range(0, len(items), per_page):
        part = items[first:first + per_page]
        columns = 1 if len(part) == 1 else 2
        rows = int(np.ceil(len(part) / columns))
        fig, axes = plt.subplots(rows, columns, figsize=(14, max(4, 3.2 * rows)), layout='constrained', squeeze=False)
        fig.suptitle(f'{title} | {first + 1}–{first + len(part)} of {len(items)}')
        for ax, (row, spectrum) in zip(axes.flat, part):
            f, a = np.asarray(spectrum['f']), np.asarray(spectrum['amplitude'])
            valid = (f > 0) & np.isfinite(a) & (a > 0)
            ax.set_xscale('log')
            if valid.any():
                display_f, display_a = _plot_spectral_envelope(f[valid], a[valid])
                ax.semilogy(display_f, display_a, color='tab:blue', linewidth=1.1, label='Native FFT')
            fp, ap = np.asarray(spectrum.get('padded_f', [])), np.asarray(spectrum.get('padded_amplitude', []))
            vp = (fp > 0) & np.isfinite(ap) & (ap > 0)
            if vp.any():
                display_f, display_a = _plot_spectral_envelope(fp[vp], ap[vp])
                ax.loglog(display_f, display_a, color='tab:orange', linewidth=0.8, alpha=0.65, label='Zero-padded FFT')
            target = row.get('target_hz', np.nan)
            if pd.notna(target) and target > 0:
                ax.axvline(target, color='black', linestyle='--', linewidth=1, alpha=0.7, label=f'Target {target:g} Hz')
            for col, label, marker in _PLOT_METHODS:
                peak = row.get(col, np.nan)
                if pd.notna(peak) and peak > 0:
                    amplitude_key = col.removesuffix('_hz') + '_amplitude'
                    yy = row.get(amplitude_key, np.nan)
                    if pd.notna(yy) and np.isfinite(yy) and yy > 0:
                        ax.scatter([peak], [yy], marker=marker, s=25, zorder=4, label=f'{label}: {peak:.6g} Hz')
            if not valid.any() and not vp.any():
                reason = str(row.get('quality_flags', '') or row.get('analysis_status', 'No positive finite spectral amplitudes'))
                ax.text(0.5, 0.5, 'Spectrum unavailable:\n' + reason.replace(';', ';\n'),
                        ha='center', va='center', fontsize=9, transform=ax.transAxes, wrap=True)
            elif _plot_timing_row(row):
                ax.text(0.02, 0.97, 'TIMING WARNING: diagnostic spectrum across timestamp gaps',
                        ha='left', va='top', fontsize=8, color='firebrick', transform=ax.transAxes,
                        bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
            _plot_style(ax, xlabel='Frequency [Hz]', ylabel=f'Peak amplitude [{_plot_unit(row)}]', title=_plot_title(row))
            _plot_legend(ax)
        for ax in axes.flat[len(part):]:
            ax.set_visible(False)
        fig.supxlabel('Long spectra: display-only max-preserving envelope (up to 6000 buckets); '
                      'peak estimation uses every bin.', fontsize=8)
        figures.append(fig)
    return figures


def plot_noise(noise_items, title):
    if not noise_items:
        return _plot_empty(title)
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), layout='constrained')
    fig.suptitle(title)
    unavailable = []
    for row, spectrum in noise_items:
        f, amplitude = np.asarray(spectrum['f']), np.asarray(spectrum['amplitude'])
        condition = row.get('noise_condition', row.get('condition', 'noise control'))
        record_label = row.get('file_label', row.get('file_id', row.get('record_id', row.get('label', ''))))
        label = f'{record_label}: {condition}'
        if _plot_timing_row(row):
            label += ' [timing warning; diagnostic]'
        good = (f > 0) & (amplitude > 0) & np.isfinite(amplitude)
        if good.any():
            display_f, display_a = _plot_spectral_envelope(f[good], amplitude[good])
            axes[0].loglog(display_f, display_a, linewidth=0.9, alpha=0.85, label=label)
        else:
            unavailable.append(f'{label}: {row.get("quality_flags", "") or row.get("analysis_status", "unavailable")}')
        psd = np.asarray(spectrum.get('psd', []))
        if psd.size == f.size:
            good_psd = (f > 0) & (psd > 0) & np.isfinite(psd)
            if good_psd.any():
                display_f, display_psd = _plot_spectral_envelope(f[good_psd], psd[good_psd])
                axes[1].loglog(display_f, display_psd, linewidth=0.9, alpha=0.85, label=label)
    for ax in axes:
        if not ax.lines:
            ax.text(0.5, 0.5, 'No positive finite spectrum values to display.', ha='center', va='center', transform=ax.transAxes)
    units = sorted(set(_plot_unit(row) for row, _ in noise_items))
    unit = units[0] if len(units) == 1 else 'recorded units (see table)'
    _plot_style(axes[0], xlabel='Frequency [Hz]', ylabel=f'Peak amplitude [{unit}]', title='Noise amplitude spectrum')
    _plot_style(axes[1], xlabel='Frequency [Hz]', ylabel=f'PSD [{unit}²/Hz]', title='Noise power spectral density')
    for ax in axes:
        _plot_legend(ax)
    warning_count = sum(_plot_timing_row(row) for row, _ in noise_items)
    notes = ['Separate control recordings; long spectra use a display-only max-preserving envelope.']
    if warning_count:
        notes.append(f'{warning_count} control(s) interpolated across timestamp gaps; strict FFT eligibility failed.')
    notes.extend(unavailable)
    fig.supxlabel('\n'.join(notes), fontsize=8, color='firebrick' if warning_count or unavailable else '0.3')
    return fig


def plot_resolution(summary):
    """Separate repeated single-tone estimation from two-tone resolving power."""
    figures = []
    if summary.empty:
        return [_plot_empty('Frequency estimation trials')]
    grouping = ['base_hz', 'window_variant'] if 'window_variant' in summary else ['base_hz']
    for keys, group in summary.groupby(grouping, sort=True, dropna=False):
        keys = keys if isinstance(keys, tuple) else (keys,)
        base = keys[0]
        variant = f' | {keys[1]}' if len(keys) > 1 else ''
        fig, axes = plt.subplots(1, 2, figsize=(14, 6), layout='constrained')
        computed = group.get('fft_computed', group.get('fft_valid', pd.Series(False, index=group.index))).fillna(False).astype(bool)
        strict = group.get('fft_valid', pd.Series(False, index=group.index)).fillna(False).astype(bool)
        ncomputed, nstrict = int(computed.sum()), int(strict.sum())
        timing_mask = _plot_timing_mask(group)
        fig.suptitle(f'{base:g} Hz trials{variant}: frequency estimation (not a two-tone resolution test)\n'
                     f'{ncomputed}/{len(group)} computed; {nstrict}/{len(group)} strictly eligible')
        target_offset = _plot_values(group, 'target_hz') - base
        for col, label, marker in _PLOT_METHODS:
            offset = _plot_values(group, col) - base
            axes[0].scatter(target_offset, offset, marker=marker, s=48, alpha=0.8, label=label)
            axes[1].scatter(target_offset, offset - target_offset, marker=marker, s=48, alpha=0.8, label=label)
            _plot_warning_points(axes[0], target_offset, offset, timing_mask)
            _plot_warning_points(axes[1], target_offset, offset - target_offset, timing_mask)
        finite = target_offset[np.isfinite(target_offset)]
        if finite.size and ncomputed:
            pad = max(0.001, np.ptp(finite) * 0.1)
            limits = (finite.min() - pad, finite.max() + pad)
            axes[0].plot(limits, limits, '--', color='0.4', label='Target')
            for ax in axes:
                ax.set_xlim(limits)
        if ncomputed:
            axes[1].axhline(0, color='0.4', linewidth=1)
        else:
            reasons = sorted(set(str(v) for v in group.get('quality_flags', group.get('analysis_status', pd.Series('unavailable', index=group.index)))))
            for ax in axes:
                ax.text(0.5, 0.5, 'No computed frequency estimates.\n' + '\n'.join(reasons),
                        ha='center', va='center', transform=ax.transAxes, fontsize=8, wrap=True)
        df = _plot_values(group, 'native_df_hz')
        finite_df = df[np.isfinite(df)]
        notes = []
        if finite_df.size:
            notes.append(f'Native bin spacing: {finite_df.min():.5g}–{finite_df.max():.5g} Hz; zero padding only refines the frequency grid.')
        if timing_mask.any():
            notes.append('Open red circles: diagnostic estimates across timestamp gaps; strict FFT eligibility failed.')
        if notes:
            fig.supxlabel('\n'.join(notes), fontsize=9)
        _plot_style(axes[0], xlabel=f'Target − {base:g} Hz [Hz]', ylabel=f'Estimate − {base:g} Hz [Hz]', title='Individual recordings; repeated observations may overlap')
        _plot_style(axes[1], xlabel=f'Target − {base:g} Hz [Hz]', ylabel='Estimate − target [Hz]', title='Individual frequency errors')
        for ax in axes:
            _plot_legend(ax)
            ax.ticklabel_format(axis='both', style='plain', useOffset=False)
        figures.append(fig)
    return figures


def plot_resolution_spectra(items):
    """Local display crop only: every peak passed here was selected globally."""
    figures = []
    groups = {}
    for row, spectrum in items:
        if pd.notna(row.get('base_hz', np.nan)):
            variant = row.get('window_variant', '')
            variant = str(variant) if pd.notna(variant) else ''
            key = (float(row['base_hz']), variant)
            groups.setdefault(key, []).append((row, spectrum))
    for (base, variant), group in sorted(groups.items()):
        fig, ax = plt.subplots(figsize=(14, 6), layout='constrained')
        ncomputed = sum(bool(row.get('fft_computed', row.get('fft_valid', False))) for row, _ in group)
        nstrict = sum(bool(row.get('fft_valid', False)) for row, _ in group)
        native_spacing = [float(row.get('native_df_hz', np.nan)) for row, _ in group]
        finite_df = [d for d in native_spacing if np.isfinite(d) and d > 0]
        half_width = max(0.04, 4 * max(finite_df, default=0.01))
        for row, spectrum in group:
            f, a = np.asarray(spectrum['f']), np.asarray(spectrum['amplitude'])
            fp, ap = np.asarray(spectrum.get('padded_f', f)), np.asarray(spectrum.get('padded_amplitude', a))
            mask = (fp > base - half_width) & (fp < base + half_width)
            if not mask.any():
                continue
            label = _plot_title(row) + (' [timing warning; diagnostic]' if _plot_timing_row(row) else '')
            line, = ax.plot(fp[mask] - base, ap[mask], linewidth=1, alpha=0.75, label=label)
            native = (f > base - half_width) & (f < base + half_width)
            ax.scatter(f[native] - base, a[native], color=line.get_color(), s=18, alpha=0.8)
        ax.set_xlim(-half_width, half_width)
        _plot_style(ax, xlabel=f'Frequency − {base:g} Hz [Hz]', ylabel=f'Peak amplitude [{_plot_unit(group[0][0])}]',
                    title=f'{base:g} Hz {variant}: diagnostic zoom; native bins marked; peak selection still uses the full positive band\n'
                          f'{ncomputed}/{len(group)} computed; {nstrict}/{len(group)} strictly eligible')
        if not ax.lines:
            reasons = sorted(set(str(row.get('quality_flags', '') or row.get('analysis_status', 'unavailable')) for row, _ in group))
            ax.text(0.5, 0.5, 'No spectra available in this display window.\n' + '\n'.join(reasons),
                    transform=ax.transAxes, ha='center', va='center', fontsize=8, wrap=True)
        _plot_legend(ax)
        figures.append(fig)
    return figures




def plot_volume_waveforms(items, title):
    """Display central raw cycles; report full-window extrema occupancy without a clipping claim."""
    if not items:
        return _plot_empty(title)
    fig, axes = plt.subplots(len(items), 1, figsize=(14, max(4, 2.7 * len(items))),
                             layout='constrained', squeeze=False)
    fig.suptitle(title + ': raw central-window waveforms')
    for ax, (row, spectrum) in zip(axes.flat, items):
        t = np.asarray(spectrum.get('raw_t', spectrum.get('t', [])), dtype=float)
        y = np.asarray(spectrum.get('raw_y', spectrum.get('y', [])), dtype=float)
        if not len(t) or len(t) != len(y):
            ax.text(0.5, 0.5, 'Raw waveform unavailable.', transform=ax.transAxes, ha='center')
            continue
        finite = np.isfinite(t) & np.isfinite(y)
        t, y = t[finite], y[finite]
        if not len(t):
            continue
        target = float(row.get('target_hz', np.nan))
        duration = min(2.0, 3.0 / target) if np.isfinite(target) and target > 0 else 2.0
        midpoint = (t[0] + t[-1]) / 2
        start, end = max(t[0], midpoint - duration / 2), min(t[-1], midpoint + duration / 2)
        selected = (t >= start) & (t <= end)
        ax.plot(t[selected], y[selected], color='tab:blue', linewidth=0.8, marker='.', markersize=2)
        minimum = float(row.get('raw_min_value', np.min(y)))
        maximum = float(row.get('raw_max_value', np.max(y)))
        fraction_min = float(row.get('raw_fraction_at_min', np.mean(y == minimum)))
        fraction_max = float(row.get('raw_fraction_at_max', np.mean(y == maximum)))
        volume = row.get('volume_setting', '?')
        ax.set_title(f'Volume {volume} | display {start:.6g}–{end:.6g} s | '
                     f'full-window min {minimum:g} ({100 * fraction_min:.3g}%); '
                     f'max {maximum:g} ({100 * fraction_max:.3g}%)', loc='left', fontsize=10)
        _plot_style(ax, xlabel='Time in recording [s]', ylabel=f'Raw signal [{_plot_unit(row)}]')
    fig.supxlabel('Original samples, unfiltered; up to three central target cycles (maximum 2 s). '
                  'Extrema occupancy describes recorded values; it does not establish calibrated sensor saturation.', fontsize=9)
    return fig


In [ ]:
# sinewave-cell: target_peak_diagnostics
import numpy as np
import pandas as pd
from scipy.signal import find_peaks


def target_peak_diagnostics(row, spectrum, settings):
    """Supplemental target-conditioned estimates/ranks; never replace global maxima."""
    result = {"target_status": "unavailable", "target_native_rank": np.nan,
              "target_padded_rank": np.nan, "target_to_strongest_padded_percent": np.nan}
    for prefix in ("target_native", "target_padded"):
        for suffix in ("_bin_hz", "_bin_amplitude", "_parabolic_hz", "_parabolic_amplitude"):
            result[prefix + suffix] = np.nan
        result[prefix + "_status"] = "unavailable"
        result[prefix + "_peak_count"] = 0
    target = row.get("target_hz", np.nan)
    if not np.isfinite(target) or not row.get("fft_computed", False):
        return result, pd.DataFrame()
    minimum = float(settings.get("minimum_hz", 0.1))
    width = settings.get("search_halfwidth_hz")
    if width is None:
        width = min(0.4 * target, max(3 * row["native_df_hz"], 0.15 * target))
    low, high = max(minimum, target - width), min(row["nyquist_hz"], target + width)
    result.update(target_search_lower_hz=low, target_search_upper_hz=high,
                  target_search_halfwidth_hz=width, target_ranking_minimum_hz=minimum,
                  target_search_definition="supplemental target band; global estimator unchanged")
    points = int(row.get("interpolation_points", 3))
    catalogs = []
    for grid, fkey, akey in (("native", "f", "amplitude"), ("padded", "padded_f", "padded_amplitude")):
        prefix = "target_" + grid
        f, a = np.asarray(spectrum[fkey]), np.asarray(spectrum[akey])
        if len(f) < 3:
            continue
        df = f[1] - f[0]
        eligible = np.flatnonzero((f >= low) & (f <= high) & (f > 0))
        if not len(eligible):
            result[prefix + "_status"] = "no_bins_in_target_band"
            continue
        k = int(eligible[np.argmax(a[eligible])])
        result[prefix + "_bin_hz"] = float(f[k])
        result[prefix + "_bin_amplitude"] = float(a[k])
        # Each interior local maximum is counted once, including window sidelobes/noise.
        indices = find_peaks(a)[0]
        indices = indices[(f[indices] >= minimum) & (f[indices] > 0) & (f[indices] < row["nyquist_hz"])]
        fitted = _quadratic_peak_fits(f, a, indices, row["interpolation_scale"], points)
        vertices, peaks = fitted["parabolic_hz"], fitted["parabolic_amplitude"]
        good = ((fitted["interpolation_status"] == "ok") & (vertices >= minimum)
                & (vertices > 0) & (vertices < row["nyquist_hz"]))
        indices, vertices, peaks = indices[good], vertices[good], peaks[good]
        ranks = 1 + len(peaks) - np.searchsorted(np.sort(peaks), peaks, side="right")
        catalog = pd.DataFrame({"grid": grid, "peak_bin_index": indices,
                                "bin_hz": f[indices], "interpolated_hz": vertices,
                                "interpolated_amplitude": peaks, "rank": ranks,
                                "is_target": False, "interpolation_points": points})
        result[prefix + "_peak_count"] = len(catalog)
        matches = np.flatnonzero(indices == k)
        status = "available"
        if k in (0, len(f)-1, eligible[0], eligible[-1]):
            status = "target_peak_at_search_boundary"
        elif not len(matches):
            status = f"target_maximum_has_no_valid_local_{points}_point_fit"
        elif not low <= vertices[matches[0]] <= high:
            status = "target_vertex_outside_search_band"
        if status == "available":
            j = int(matches[0])
            result[prefix + "_parabolic_hz"] = float(vertices[j])
            result[prefix + "_parabolic_amplitude"] = float(peaks[j])
            result[prefix + "_rank"] = int(ranks[j])
            catalog.loc[j, "is_target"] = True
            if grid == "padded":
                result["target_to_strongest_padded_percent"] = 100 * float(peaks[j] / peaks.max())
        result[prefix + "_status"] = status
        catalogs.append(catalog)
    result["target_status"] = result["target_padded_status"]
    return result, pd.concat(catalogs, ignore_index=True) if catalogs else pd.DataFrame()


In [ ]:
# sinewave-cell: sweep_visuals
"""Notebook plotting helpers: supplied arrays only, no file reads or estimators."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal


_V3_FREQUENCY_METHODS = (
    ('global_bin_hz', 'Native FFT maximum', 'o'),
    ('global_parabolic_hz', 'Native quadratic fit', 's'),
    ('global_padded_hz', 'Padded FFT maximum', '^'),
    ('global_padded_parabolic_hz', 'Padded quadratic fit', 'D'),
)


def _v3_values(frame, key):
    return pd.to_numeric(frame.get(key, pd.Series(np.nan, index=frame.index)),
                         errors='coerce').to_numpy(dtype=float)


def _v3_unit(frame):
    units = frame.get('unit', 'recorded units')
    if isinstance(units, pd.Series):
        unique = units.dropna().astype(str).unique()
        return unique[0] if len(unique) == 1 else 'recorded units'
    return str(units)


def _v3_style(ax, xlabel='', ylabel='', title=''):
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=11)
    ax.grid(True, which='major', alpha=0.23)
    ax.set_axisbelow(True)


def _v3_legend(ax):
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        unique = dict(zip(labels, handles))
        ax.legend(unique.values(), unique.keys(), fontsize=8, frameon=True)


def _v3_empty(title):
    fig, ax = plt.subplots(figsize=(12, 4), layout='constrained')
    ax.text(0.5, 0.5, 'No selected data available.', ha='center', transform=ax.transAxes)
    ax.set_axis_off()
    fig.suptitle(title)
    return fig


def _v3_qc_note(frame):
    computed = frame.get('fft_computed', frame.get('fft_valid', pd.Series(False, index=frame.index)))
    strict = frame.get('fft_valid', pd.Series(False, index=frame.index))
    warning = computed.fillna(False).astype(bool) & ~strict.fillna(False).astype(bool)
    missing = (~computed.fillna(False).astype(bool)).sum()
    notes = []
    if warning.any():
        notes.append(f'Timing: {int(warning.sum())}/{len(frame)} diagnostic FFT windows; details in the QC table.')
    if missing:
        notes.append(f'{int(missing)} window(s) without an FFT estimate.')
    return ' '.join(notes)


def _v3_footer(fig, note):
    if note:
        fig.supxlabel(note, fontsize=9, color='0.3')


def _v3_envelope(t, y, width_s):
    """Display only: preserve minimum and maximum of every occupied time bin."""
    bins = np.floor((t - t[0]) / width_s).astype(np.int64)
    first = np.r_[0, np.flatnonzero(np.diff(bins)) + 1]
    last = np.r_[first[1:] - 1, len(t) - 1]
    return ((t[first] + t[last]) / 2,
            np.minimum.reduceat(y, first), np.maximum.reduceat(y, first))


def _v3_timing(t, max_gap_over_median):
    if not np.isfinite(max_gap_over_median) or max_gap_over_median <= 0:
        raise ValueError('max_gap_over_median must be finite and positive.')
    if len(t) < 3 or not np.isfinite(t).all() or np.any(np.diff(t) <= 0):
        return None
    dt = np.diff(t)
    median_dt = float(np.median(dt))
    threshold = max_gap_over_median + max(1e-9, abs(max_gap_over_median) * 1e-9)
    breaks = np.flatnonzero(dt / median_dt > threshold) + 1
    return {'breaks': breaks, 'max_ratio': float(np.max(dt) / median_dt),
            'max_gap_s': float(np.max(dt)), 'median_dt_s': median_dt}


def plot_record_overview(record, schedule, label, max_gap_over_median=3.0, gap_policy='diagnostic'):
    """Original raw envelope plus window-centered display; no resampling or filtering."""
    t, y = np.asarray(record['t'], float), np.asarray(record['y'], float)
    unit = str(record.get('unit', 'recorded units'))
    if gap_policy not in ('diagnostic', 'strict'):
        raise ValueError("gap_policy must be 'diagnostic' or 'strict'.")
    timing = _v3_timing(t, max_gap_over_median)
    if timing is None or len(t) != len(y) or not np.isfinite(y).all():
        return _v3_empty(f'{label}: invalid raw values or timestamps')
    fig, axes = plt.subplots(2, 1, figsize=(14, 7.3), sharex=True, layout='constrained')
    fig.suptitle(f'{label}: raw acquisition and analysis windows')
    width = max(0.02, float(t[-1] - t[0]) / 6000)
    for j, indices in enumerate(np.split(np.arange(len(t)), timing['breaks'])):
        if not len(indices):
            continue
        tx, lo, hi = _v3_envelope(t[indices], y[indices], width)
        axes[0].vlines(tx, lo, hi, linewidth=0.6, color='tab:blue', alpha=0.85,
                       label='Raw min/max envelope' if j == 0 else None)
    n_windows = 0
    if schedule is not None and not schedule.empty:
        for _, row in schedule.iterrows():
            if not bool(row.get('selected', True)):
                continue
            a, b = row.get('analysis_start_s', np.nan), row.get('analysis_end_s', np.nan)
            if not pd.notna(a) or not pd.notna(b) or not np.isfinite(a) or not np.isfinite(b):
                continue
            mask = (t >= a) & (t < b)
            if not mask.any():
                continue
            axes[0].axvspan(a, b, color='tab:green', alpha=0.1,
                           label='Analysis window' if n_windows == 0 else None)
            centered = y[mask] - np.mean(y[mask])
            tx, lo, hi = _v3_envelope(t[mask], centered, width)
            axes[1].vlines(tx, lo, hi, linewidth=0.6, color='tab:blue', alpha=0.85)
            axes[1].axvspan(a, b, color='tab:green', alpha=0.07)
            if len(schedule) <= 12:
                volume = row.get('volume_setting', np.nan)
                target = row.get('target_hz', np.nan)
                tag = (f'Vol {volume:g}' if pd.notna(volume) else
                       f'{target:g} Hz' if pd.notna(target) else str(row.get('role', '')))
                axes[0].text((a + b) / 2, 0.98, tag, ha='center', va='top', fontsize=9,
                             transform=axes[0].get_xaxis_transform())
            n_windows += 1
    if not n_windows:
        axes[1].text(0.5, 0.5, 'No finite analysis windows selected.', ha='center', transform=axes[1].transAxes)
    axes[1].axhline(0, color='0.5', linewidth=0.7)
    _v3_style(axes[0], ylabel=f'Raw signal [{unit}]', title='Recorded baseline and excursions')
    _v3_style(axes[1], xlabel='Time from first recorded sample [s]', ylabel=f'Signal − window mean [{unit}]',
              title='Selected windows, each centered separately (display only)')
    _v3_legend(axes[0])
    note = f'Original samples; min/max display bins {width:.3g} s. Centering changes only the lower-panel display.'
    if len(timing['breaks']):
        note += f" Timing: {len(timing['breaks'])} gaps > {max_gap_over_median:g} × median interval; see timestamp table."
    _v3_footer(fig, note)
    return fig


def plot_spectrogram(record, schedule, label, settings=None):
    """Display-only PSD; nominal dots at segment centers; identical colors for either axis scale."""
    settings = {} if settings is None else settings
    # Accept either the plot-settings dictionary or the complete notebook config.
    options = dict(settings.get('plots', settings))
    for key, value in settings.get('fft', {}).items():
        options.setdefault(key, value)
    t, y = np.asarray(record['t'], float), np.asarray(record['y'], float)
    gap_limit = float(options.get('max_gap_over_median', 3.0))
    policy = options.get('gap_policy', 'diagnostic')
    if policy not in ('diagnostic', 'strict'):
        raise ValueError("gap_policy must be 'diagnostic' or 'strict'.")
    timing = _v3_timing(t, gap_limit)
    if timing is None or len(t) != len(y) or not np.isfinite(y).all():
        return _v3_empty(f'{label}: invalid values or timestamps; spectrogram unavailable')
    fig, ax = plt.subplots(figsize=(14, 6.2), layout='constrained')
    fig.suptitle(f'{label}: spectrogram and nominal frequency schedule')
    if len(timing['breaks']) and policy == 'strict':
        ax.text(0.5, 0.5, 'Spectrogram unavailable under the configured strict timing rule.\n'
                f"{len(timing['breaks'])} gap(s) exceed {gap_limit:g} × the median interval.",
                ha='center', va='center', transform=ax.transAxes)
        _v3_style(ax, xlabel='Time from first recorded sample [s]', ylabel='Frequency [Hz]')
        return fig
    fs = (len(t) - 1) / (t[-1] - t[0])
    tu = np.linspace(t[0], t[-1], len(t))
    yu = np.interp(tu, t, y)
    segment_s = float(options.get('spectrogram_segment_s', 2.0))
    overlap = float(options.get('spectrogram_overlap', 0.75))
    dynamic_range = float(options.get('spectrogram_dynamic_range_db', 100.0))
    scale = options.get('spectrogram_frequency_scale', 'linear')
    if not np.isfinite(segment_s) or segment_s <= 0 or not 0 <= overlap < 1:
        raise ValueError('Spectrogram segment_s must be positive; overlap must be in [0, 1).')
    if not np.isfinite(dynamic_range) or dynamic_range <= 0 or scale not in ('linear', 'log'):
        raise ValueError('Spectrogram dynamic range must be positive; frequency scale must be linear or log.')
    fig.suptitle(f'{label}: spectrogram ({scale} frequency) and nominal settings')
    nperseg = min(len(t), max(8, int(round(segment_s * fs))))
    noverlap = min(nperseg - 1, int(round(overlap * nperseg)))
    f, st, psd = signal.spectrogram(yu, fs=fs, window='hann', nperseg=nperseg,
                                  noverlap=noverlap, detrend='constant', scaling='density', mode='psd')
    # Both displays use the same full PSD and color normalization. The logarithmic
    # view omits only the zero-frequency row, which cannot appear on a log axis.
    all_power_db = 10 * np.log10(np.maximum(psd, np.finfo(float).tiny))
    peak_db = float(np.max(all_power_db))
    frequency_mask = f > 0 if scale == 'log' else np.ones(len(f), dtype=bool)
    power_db = all_power_db[frequency_mask]
    mesh = ax.pcolormesh(st + tu[0], f[frequency_mask], power_db, shading='auto',
                        cmap=options.get('spectrogram_cmap', 'viridis'),
                        vmin=peak_db - dynamic_range, vmax=peak_db, rasterized=True)
    unit = str(record.get('unit', 'recorded units'))
    cb = fig.colorbar(mesh, ax=ax, pad=0.02)
    cb.set_label(f'PSD [dB re 1 ({unit})²/Hz]')
    nominal_centers, nominal_frequencies = [], []
    if schedule is not None and not schedule.empty:
        # Nominal bounds, including deselected analysis frequencies, preserve acquisition mapping.
        for _, row in schedule.iterrows():
            target, a, b = (row.get(key, np.nan) for key in ('target_hz', 'nominal_start_s', 'nominal_end_s'))
            if any(pd.isna(v) for v in (target, a, b)) or not all(np.isfinite(v) for v in (target, a, b)):
                continue
            if target <= 0 or b <= a:
                continue
            nominal_centers.append((a + b) / 2)
            nominal_frequencies.append(target)
    if nominal_centers:
        marker_size = float(options.get('spectrogram_nominal_size', 18.0))
        if not np.isfinite(marker_size) or marker_size <= 0:
            raise ValueError('spectrogram_nominal_size must be finite and positive (points squared).')
        ax.scatter(nominal_centers, nominal_frequencies,
                   marker=options.get('spectrogram_nominal_marker', 'o'),
                   color=options.get('spectrogram_nominal_color', 'cyan'),
                   s=marker_size, linewidths=0, alpha=0.95, zorder=4,
                   label='Nominal settings (segment centers)')
    limits = options.get('spectrogram_frequency_limits_hz', [0.0, None])
    if limits is None:
        limits = [0.0, None]
    if len(limits) != 2:
        raise ValueError('spectrogram_frequency_limits_hz needs [minimum, maximum].')
    lower = 0.0 if limits[0] is None else float(limits[0])
    upper = fs / 2 if limits[1] is None else float(limits[1])
    if scale == 'log' and lower <= 0:
        lower = float(f[f > 0][0])
    if not np.isfinite(lower) or not np.isfinite(upper) or lower < 0 or upper <= lower:
        raise ValueError('Invalid spectrogram frequency limits.')
    ax.set_yscale(scale)
    ax.set_ylim(lower, upper)
    ax.set_xlim(t[0], t[-1])
    _v3_style(ax, xlabel='Time from first recorded sample [s]', ylabel='Frequency [Hz]')
    ax.grid(False)
    if nominal_centers:
        ax.legend(loc='upper left', fontsize=8, frameon=True)
    note = (f'Display only: fixed-N linear interpolation; fs={fs:.6g} Hz; periodic Hann {nperseg/fs:.3g} s; '
            f'{100*noverlap/nperseg:.1f}% overlap; constant detrend; {dynamic_range:g} dB color range.')
    if scale == 'log':
        note += f' Log view begins at {lower:.3g} Hz; zero frequency is omitted.'
    if len(timing['breaks']):
        note += (f"\nTiming: interpolates across {len(timing['breaks'])} gaps > {gap_limit:g} × median; "
                 f"largest gap {1e3*timing['max_gap_s']:.3g} ms. See timestamp table.")
    _v3_footer(fig, note)
    return fig


def plot_sweep(summary, title, settings=None):
    """Global estimators in all panels; a labeled error zoom preserves the full-error panel."""
    if summary.empty:
        return _v3_empty(title)
    settings = {} if settings is None else settings.get('plots', settings)
    x = _v3_values(summary, 'target_hz')
    valid_x = np.isfinite(x) & (x > 0)
    finite_x = x[valid_x]
    monotonic = len(finite_x) > 0 and (np.all(np.diff(finite_x) >= 0) or np.all(np.diff(finite_x) <= 0))
    line = '-' if monotonic else 'None'
    fig, axes = plt.subplots(2, 2, figsize=(14, 8.5), layout='constrained')
    fig.suptitle(title)
    zoom = settings.get('frequency_error_zoom_hz', [-0.15, 0.15])
    if len(zoom) != 2 or not all(np.isfinite(v) for v in zoom) or zoom[1] <= zoom[0]:
        raise ValueError('frequency_error_zoom_hz needs two increasing finite values.')
    outside = np.zeros(len(summary), dtype=bool)
    for key, label, marker in _V3_FREQUENCY_METHODS:
        y = _v3_values(summary, key)
        good = valid_x & np.isfinite(y)
        shown = np.where(good, y, np.nan)
        axes[0, 0].plot(x, shown, marker=marker, linestyle=line, markersize=4, linewidth=1, label=label)
        for ax in (axes[0, 1], axes[1, 0]):
            ax.plot(x, shown - x, marker=marker, linestyle=line, markersize=4, linewidth=1, label=label)
        outside |= good & ((y - x < zoom[0]) | (y - x > zoom[1]))
    axes[0, 0].plot(x, np.where(valid_x, x, np.nan), '--', color='black', linewidth=1, label='Nominal setting')
    axes[0, 0].set_yscale('log')
    for ax in (axes[0, 1], axes[1, 0]):
        ax.axhline(0, linewidth=0.8, color='0.4')
    axes[1, 0].set_ylim(*zoom)
    axes[1, 0].text(0.02, 0.98, f'{int(outside.sum())} frequency setting(s) outside this zoom; see full error above.',
                    transform=axes[1, 0].transAxes, va='top', fontsize=8, color='0.35')
    samples = np.divide(_v3_values(summary, 'fs_hz'), x, out=np.full_like(x, np.nan), where=valid_x)
    axes[1, 1].plot(x, samples, 'o-', markersize=4, label='Timestamp-derived fs / nominal frequency')
    axes[1, 1].axhline(2, color='crimson', linestyle='--', linewidth=1, label='2 samples per cycle')
    axes[1, 1].set_yscale('log')
    specs = [
        (axes[0, 0], 'Estimated frequency [Hz]', 'Global positive-frequency maximum'),
        (axes[0, 1], 'Estimate − setting [Hz]', 'Frequency error: full scale'),
        (axes[1, 0], 'Estimate − setting [Hz]', 'Frequency error: configured limits'),
        (axes[1, 1], 'Samples per nominal cycle', 'Sampling margin'),
    ]
    for ax, ylabel, subtitle in specs:
        ax.set_xscale('log')
        _v3_style(ax, xlabel='Nominal sine frequency [Hz]', ylabel=ylabel, title=subtitle)
    _v3_legend(axes[0, 0])
    _v3_legend(axes[1, 1])
    _v3_footer(fig, 'All four estimates search the full positive-frequency spectrum. ' + _v3_qc_note(summary))
    return fig


def plot_sweep_amplitudes(summary, title):
    """Separate log-log panels preserve distinct spectral, RMS, and peak-to-peak definitions."""
    if summary.empty:
        return _v3_empty(title)
    x, unit = _v3_values(summary, 'target_hz'), _v3_unit(summary)
    finite_x = x[np.isfinite(x)]
    line = '-' if np.all(np.diff(finite_x) >= 0) or np.all(np.diff(finite_x) <= 0) else 'None'
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), layout='constrained')
    fig.suptitle(title)
    series = [
        (axes[0], 'global_padded_parabolic_amplitude', 'Global peak: padded quadratic fit', 'o'),
        (axes[1], 'raw_rms', 'Demeaned RMS', 'o'),
        (axes[2], 'raw_peak_to_peak', 'Raw peak-to-peak', 'o'),
    ]
    if 'target_padded_parabolic_amplitude' in summary:
        series.insert(1, (axes[0], 'target_padded_parabolic_amplitude', 'Target-near peak (supplemental)', 'x'))
    for ax, key, label, marker in series:
        values = _v3_values(summary, key)
        good = np.isfinite(values) & (values > 0) & np.isfinite(x) & (x > 0)
        ax.plot(x, np.where(good, values, np.nan), marker=marker, linestyle=line,
                markersize=4, linewidth=1.2, label=label)
    subtitles = ['Spectral peak amplitude', 'Demeaned RMS: all recorded components', 'Raw peak-to-peak: extrema']
    for ax, subtitle in zip(axes, subtitles):
        ax.set_xscale('log')
        ax.set_yscale('log')
        _v3_style(ax, xlabel='Nominal sine frequency [Hz]', ylabel=unit, title=subtitle)
    _v3_legend(axes[0])
    _v3_footer(fig, 'Logarithmic amplitude axes; no amplitude metric replaces another. ' + _v3_qc_note(summary))
    return fig


def plot_peak_ranking(summary, title):
    """Supplemental target-near local-peak ranks; explicitly not an SNR estimate."""
    if summary.empty:
        return _v3_empty(title)
    y = np.arange(len(summary))
    fig, axes = plt.subplots(1, 2, figsize=(14, max(4.5, 0.33 * len(summary) + 1.8)),
                             sharey=True, layout='constrained', gridspec_kw={'width_ratios': [1, 1]})
    rank_title = title if 'target-peak ranks' in title.lower() else title + ': target-peak ranks'
    fig.suptitle(rank_title if 'not snr' in rank_title.lower() else rank_title + ' (not SNR)')
    labels = []
    for i, (_, row) in enumerate(summary.iterrows()):
        target = row.get('target_hz', np.nan)
        step = row.get('segment_index', i + 1)
        labels.append(f'{int(step):02d} | {float(target):g} Hz')
    for key, name, offset, marker in [('target_native_rank', 'Native quadratic fit', -0.11, 'o'),
                                      ('target_padded_rank', 'Padded quadratic fit', 0.11, 'x')]:
        ranks = _v3_values(summary, key)
        valid = np.isfinite(ranks) & (ranks > 0)
        axes[0].scatter(ranks[valid], y[valid] + offset, s=28, marker=marker, label=name)
    percent = _v3_values(summary, 'target_to_strongest_padded_percent')
    valid = np.isfinite(percent) & (percent >= 0)
    axes[1].barh(y[valid], percent[valid], height=0.7, alpha=0.75)
    for yi, value in zip(y[valid], percent[valid]):
        axes[1].text(value + 1.0, yi, f'{value:.2f}%', va='center', fontsize=8)
    axes[0].set_xscale('log')
    axes[0].axvline(1, color='0.5', linestyle='--', linewidth=1)
    axes[0].set_yticks(y, labels)
    axes[0].invert_yaxis()
    axes[1].set_xlim(0, max(120, 1.18 * float(np.max(percent[valid])) if valid.any() else 120))
    _v3_style(axes[0], xlabel='Amplitude rank among positive-frequency local peaks (1 = largest)',
              ylabel='Acquisition step | nominal frequency')
    _v3_style(axes[1], xlabel='Target-near peak / strongest local peak [%]')
    _v3_legend(axes[0])
    unavailable = int((~valid).sum())
    note = 'Target-near selection is supplemental and does not establish detection; ranks and relative amplitudes are not SNR.'
    if unavailable:
        note += f' {unavailable} target estimate(s) unavailable; see target_status in CSV.'
    _v3_footer(fig, note)
    return fig


In [ ]:
# sinewave-cell: resolution_visuals
# sinewave-cell: resolution_visuals_v3
import numpy as np
import matplotlib.pyplot as plt
from resolution_frequency_comparison import RESOLUTION_METHODS


def _resolution_number(row, key, default=np.nan):
    try:
        return float(row.get(key, default))
    except (TypeError, ValueError):
        return float(default)


def _resolution_record_label(row):
    return str(row.get('file_label', row.get('file_id', row.get('record_id', row.get('file', 'record')))))


def plot_resolution_comparison(items, title_prefix=''):
    """Compare independent single-tone records; FFT estimators select global peaks; sine fit refines their observed neighborhood.

    A base-centered crop and per-record crop normalization are DISPLAY ONLY.
    Every finite stored global estimate is plotted, even when it is far outside
    that crop. Missing estimates and out-of-crop global peaks are listed.
    """
    groups = {}
    for row, spectrum in items:
        base = _resolution_number(row, 'base_hz')
        if np.isfinite(base):
            key = (base, str(row.get('window_variant', '')))
            groups.setdefault(key, []).append((row, spectrum))
    figures = []
    for (base, variant), group in sorted(groups.items()):
        group = sorted(group, key=lambda item: (_resolution_number(item[0], 'target_hz'), _resolution_record_label(item[0])))
        targets = sorted(set(_resolution_number(row, 'target_hz') for row, _ in group
                             if np.isfinite(_resolution_number(row, 'target_hz'))))
        palette = plt.get_cmap('tab10')
        colors = {target: palette(i % 10) for i, target in enumerate(targets)}
        spacings = [_resolution_number(row, 'native_df_hz') for row, _ in group]
        spacings = [v for v in spacings if np.isfinite(v) and v > 0]
        halfwidth = max(0.04, 4 * max(spacings, default=0.01))
        xlower, xupper = base - halfwidth, base + halfwidth
        scales = sorted(set(str(row.get('interpolation_scale', 'log')) for row, _ in group))
        fit_label = scales[0] if len(scales) == 1 else 'configured'
        pads = sorted(set(int(row.get('pad_factor', 8)) for row, _ in group))
        pad_label = f'{pads[0]}x' if len(pads) == 1 else ''
        methods = [(name + '_hz', label.replace('Padded', f'{pad_label} padded')
                    .replace('padded bin', f'{pad_label} padded bin')
                    .replace('point fit', f'point {fit_label} fit')) for name, label in RESOLUTION_METHODS]
        fig, axes = plt.subplots(1, 2, figsize=(19, 6.4), gridspec_kw={'width_ratios': [1, 1.45]})
        notes, missing, ids = [], [], []
        for ti, target in enumerate(targets):
            same_target = [(row, spec) for row, spec in group if _resolution_number(row, 'target_hz') == target]
            c = colors[target]
            for ri, (row, spec) in enumerate(same_target):
                label = _resolution_record_label(row)
                ids.append(f'{label}: {target:g} Hz')
                fp = np.asarray(spec.get('padded_f', []), dtype=float)
                ap = np.asarray(spec.get('padded_amplitude', []), dtype=float)
                if fp.shape != ap.shape:
                    raise ValueError(f'{label}: padded frequency/amplitude shape mismatch.')
                keep = (fp >= xlower) & (fp <= xupper) & np.isfinite(ap)
                if keep.any() and np.max(ap[keep]) > 0:
                    axes[0].plot(fp[keep], ap[keep] / np.max(ap[keep]), color=c,
                                 alpha=.82, lw=1.6, ls=('-', '--', ':', '-.')[ri % 4],
                                 label=f'{target:g} Hz ({len(same_target)} record' + ('s)' if len(same_target) != 1 else ')') if ri == 0 else None)
                else:
                    notes.append(f'{label}: no usable spectrum in display crop')
                for global_key in ('global_bin_hz', 'global_padded_hz'):
                    global_value = _resolution_number(row, global_key)
                    if np.isfinite(global_value) and not xlower <= global_value <= xupper:
                        notes.append(f'{label}: {global_key.removesuffix("_hz")} = {global_value:.8g} Hz, outside spectrum crop')
                offset = (ti - (len(targets) - 1) / 2) * .17
                offset += (ri - (len(same_target) - 1) / 2) * .045
                for mi, (key, _) in enumerate(methods):
                    value = _resolution_number(row, key)
                    if np.isfinite(value):
                        axes[1].scatter(mi + offset, (value - base) * 1000, color=c,
                                        s=37, marker=('o', 's', '^', 'D')[ri % 4], alpha=.9)
                    else:
                        missing.append(f'{label}/{mi + 1}')
            axes[0].axvline(target, c=c, ls=':', alpha=.55)
            axes[1].axhline((target - base) * 1000, c=c, ls=':', alpha=.5)
        axes[0].set(xlim=(xlower, xupper), ylim=(0, 1.07), xlabel='Frequency (Hz)',
                    ylabel='Normalized Hann peak amplitude',
                    title='Overlaid spectra from separate recordings')
        axes[1].set(xticks=np.arange(len(methods)), xticklabels=[label for _, label in methods],
                    xlim=(-.5, len(methods) - .5), ylabel=f'Estimated frequency - {base:g} Hz (mHz)',
                    title='Every recording and frequency estimator')
        axes[1].tick_params(axis='x', labelsize=9)
        # Deliberately autoscale all y values: no global peak is silently clipped.
        axes[1].margins(y=.1)
        for ax in axes:
            ax.grid(alpha=.23)
        axes[0].ticklabel_format(axis='x', style='plain', useOffset=False)
        axes[1].ticklabel_format(axis='y', style='plain', useOffset=False)
        if axes[0].lines:
            axes[0].legend(fontsize=9, loc='best')
        heading = f'{base:g} Hz frequency estimation'
        if variant:
            heading += f' | {variant}'
        fig.suptitle(f'{title_prefix}: {heading}' if title_prefix else heading)
        footer = ['Spectrum normalized per recording within the displayed crop; this changes display only.',
                  'Timestamp checks: diagnostics table; independent recordings, not a two-tone test.',
                  'Primary FFT estimate: padded 3-point. Original-time sine fit assumes recorded timestamps represent sampling instants; no clock correction.']
        if spacings:
            footer.insert(1, f'Native bin spacing: {min(spacings) * 1000:.4g}-{max(spacings) * 1000:.4g} mHz; padding refines the grid only.')
        if len(ids) <= 10:
            footer.append('Records: ' + '; '.join(ids))
        if missing:
            footer.append('Unavailable estimates (record/method 1-7): ' + ', '.join(missing))
        footer.extend(dict.fromkeys(notes))
        note_text = '\n'.join(footer)
        foot_height = .022 * len(footer)
        fig.text(.02, .015, note_text, ha='left', va='bottom', fontsize=8)
        fig.tight_layout(rect=(0, foot_height + .035, 1, .94))
        figures.append(fig)
    if not figures:
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.text(.5, .5, 'No resolution recordings available.', transform=ax.transAxes, ha='center')
        ax.set_axis_off()
        figures.append(fig)
    return figures


def plot_interpolation_demo(row, spectrum):
    """Inspect the configured number of actual native bins around the GLOBAL positive maximum.

    Supply FULL spectra, not the base-centered crop used in resolution overlays.
    No estimate is recomputed or replaced in the analysis table. The fitted curve
    uses the exact configured amplitude domain and is shown only for a valid fit.
    """
    f = np.asarray(spectrum.get('f', []), dtype=float)
    a = np.asarray(spectrum.get('amplitude', []), dtype=float)
    pf = np.asarray(spectrum.get('padded_f', []), dtype=float)
    pa = np.asarray(spectrum.get('padded_amplitude', []), dtype=float)
    fig, ax = plt.subplots(figsize=(10.5, 5.1))
    label = _resolution_record_label(row)
    points = int(row.get('interpolation_points', 3))
    radius = points // 2
    ax.set(xlabel='Frequency (Hz)', ylabel='Sinusoidal peak amplitude (recorded counts)',
           title=f'{label}: {points}-point quadratic fit on the spectral curve')
    notes = []
    positive = np.flatnonzero(f > 0)
    if len(f) != len(a) or not len(positive) or not np.all(np.isfinite(a[positive])) or np.max(a[positive]) <= 0:
        ax.text(.5, .5, 'No valid positive-frequency spectrum.', transform=ax.transAxes, ha='center')
        return fig
    k = int(positive[np.argmax(a[positive])])
    if not np.isclose(f[0], 0, rtol=0, atol=1e-12):
        ax.text(.5, .5, 'Full spectrum required; received a cropped spectrum.', transform=ax.transAxes, ha='center')
        return fig
    stored_bin = _resolution_number(row, 'global_bin_hz')
    if not np.isfinite(stored_bin) or not np.isclose(stored_bin, f[k], rtol=1e-11, atol=1e-12):
        ax.text(.5, .5, 'Stored global bin does not match the supplied full spectrum.', transform=ax.transAxes, ha='center')
        return fig
    lo, hi = max(0, k - radius), min(len(f), k + radius + 1)
    native_x, native_y = f[lo:hi], a[lo:hi]
    if len(pf) == len(pa):
        select = (pf >= native_x[0]) & (pf <= native_x[-1])
        ax.plot(pf[select], pa[select], '.-', ms=5, color='tab:blue',
                label=f'{int(row.get("pad_factor", 8))}x padded FFT samples')
    ax.plot(native_x, native_y, 'o', ms=8, color='tab:green', label=f'{len(native_x)} actual native FFT bins', zorder=5)
    scale = str(row.get('interpolation_scale', 'log'))
    status = str(row.get('global_interpolation_status', 'unavailable'))
    fitted = _quadratic_peak_fits(f, a, [k], scale, points)
    valid = status == 'ok' and fitted['interpolation_status'][0] == 'ok'
    if valid:
        df = f[k + 1] - f[k]
        x = np.linspace(native_x[0], native_x[-1], 301)
        curve = np.polyval(fitted['coefficients'][0], (x - f[k]) / df)
        if scale == 'log':
            curve = np.exp(curve)
        ax.plot(x, curve, '--', color='tab:orange',
                label=f'Quadratic least-squares fit in {scale} amplitude ({points} native bins)')
    if not valid:
        notes.append(f'Native fit unavailable: {status}; no substitute peak is selected.')
    for key, fit_status_key, color, line_label in (
            ('global_parabolic_hz', 'global_interpolation_status', 'tab:orange', f'Native {points}-point peak'),
            ('global_padded_parabolic_hz', 'global_padded_interpolation_status', 'tab:blue', f'Padded {points}-point peak')):
        peak = _resolution_number(row, key)
        fit_status = str(row.get(fit_status_key, 'unavailable'))
        if fit_status == 'ok' and np.isfinite(peak) and native_x[0] <= peak <= native_x[-1]:
            if key != 'global_parabolic_hz' or valid:
                ax.axvline(peak, ls=':', color=color, label=f'{line_label}: {peak:.8f} Hz')
        elif np.isfinite(peak):
            notes.append(f'{line_label}: {peak:.8f} Hz is outside this native-peak zoom (status: {fit_status}).')
        elif key == 'global_padded_parabolic_hz':
            notes.append(f'Padded fit unavailable: {fit_status}.')
    padded_bin = _resolution_number(row, 'global_padded_hz')
    if np.isfinite(padded_bin) and not native_x[0] <= padded_bin <= native_x[-1]:
        notes.append(f'Padded global maximum is at {padded_bin:.8f} Hz, outside this zoom.')
    target = _resolution_number(row, 'target_hz')
    if np.isfinite(target):
        notes.append(f'Nominal setting: {target:g} Hz. Native global maximum: {f[k]:.8f} Hz; selection uses the full positive spectrum.')
    notes.append('Timestamp checks: diagnostics table; padding and interpolation refine estimates, not physical resolving power.')
    ax.ticklabel_format(axis='x', style='plain', useOffset=False)
    ax.grid(alpha=.23)
    ax.legend(fontsize=8, loc='best')
    fig.text(.02, .015, '\n'.join(notes), ha='left', va='bottom', fontsize=8)
    fig.tight_layout(rect=(0, .03 + .025 * len(notes), 1, 1))
    return fig


In [ ]:
# sinewave-cell: volume_reference_helpers
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


VOLUME_REFERENCE_METHODS = {
    'pairing': 'Explicit configured file pairs, joined by amplifier volume setting only; separate acquisitions.',
    'amplitude': 'Global positive-frequency padded FFT maximum refined using its two neighboring bins; original spectrum preprocessing retained.',
    'normalization': 'Each sensor amplitude divided by its own amplitude at the configured reference volume; no cross-sensor unit conversion.',
    'ratio': 'IM Data28 peak amplitude / accelerometer peak acceleration in g (nominal per-record full-scale conversion); descriptive conditional proxy, not calibrated sensitivity.',
    'ratio_usable': 'A numerical comparison with confirmed matching conditions, one row per sensor and volume, finite positive amplitudes, computed FFTs, matching nominal frequency, compatible global peaks, and accelerometer extrema occupancy below threshold. Timestamp-warning estimates remain diagnostic.',
    'plateau_flag': 'Fraction of retained raw accelerometer samples equal to the observed minimum OR maximum >= configured threshold; diagnostic repeated-extrema flag, not proof of sensor saturation.',
    'limits': 'Sequential acquisitions may differ because of drift; no synchronized transfer function, phase, coherence, or time/pulse pairing. No calibration or regression inferred.',
}


def _vr_num(row, key):
    try:
        value = float(row.get(key, np.nan))
    except (TypeError, ValueError):
        return np.nan
    return value if np.isfinite(value) else np.nan


def _vr_bool(row, key, default=False):
    value = row.get(key, default)
    return bool(value) if pd.notna(value) else default


def _vr_file_mask(table, label):
    """Accept explicit record labels or full paths, never a substring match."""
    label = str(label)
    mask = pd.Series(False, index=table.index)
    for key in ('file_label', 'file_id', 'path'):
        if key in table:
            values = table[key].fillna('').astype(str)
            mask |= values.eq(label)
            if key == 'path' and '/' not in label and '\\' not in label:
                mask |= values.map(lambda value: Path(value).stem).eq(Path(label).stem)
    return mask


def _vr_occupancy(row):
    value = _vr_num(row, 'raw_extrema_occupancy_fraction')
    if np.isfinite(value):
        return value
    low, high = _vr_num(row, 'raw_fraction_at_min'), _vr_num(row, 'raw_fraction_at_max')
    return min(1.0, low + high) if np.isfinite(low) and np.isfinite(high) else np.nan


def build_volume_reference(summary, config):
    """Return one row per explicit file pair / volume; never silently average duplicates.

    config keys: pairs=[{'im_file':'IM_32', 'accel_file':'accel_4'}],
    conditions_confirmed=True, reference_level=10,
    amplitude_key='global_padded_parabolic_amplitude',
    plateau_fraction_threshold=0.01. The default frequency compatibility limit is
    2 * max(native_df_im, native_df_accel), used for both measured-peak agreement
    and measured-peak proximity to the equal nominal drive frequencies.

    Rows with unknown/invalid inputs remain visible, with ratio_usable=False.
    ratio_raw_counts_per_g is merely numerical and retains flagged values. The plotted
    selected ratio is ratio_usable_counts_per_g. A computed timing-warning spectrum is
    retained as explicitly diagnostic, not rejected again by fft_valid.
    """
    c = dict(config.get('volume_reference', config))
    pairs = c.get('pairs', [])
    if not isinstance(pairs, list) or not pairs:
        raise ValueError('volume_reference.pairs must explicitly name IM / accelerometer files.')
    seen = set()
    for pair in pairs:
        if not isinstance(pair, dict) or not pair.get('im_file') or not pair.get('accel_file'):
            raise ValueError('Each volume reference pair needs im_file and accel_file.')
        identity = (str(pair['im_file']), str(pair['accel_file']))
        if identity in seen:
            raise ValueError('Duplicate volume reference pair; it is not another repeat.')
        seen.add(identity)
    amplitude_key = str(c.get('amplitude_key', 'global_padded_parabolic_amplitude'))
    if not amplitude_key.endswith('_amplitude'):
        raise ValueError('amplitude_key must explicitly identify a spectral amplitude estimator.')
    frequency_key = amplitude_key.removesuffix('_amplitude') + '_hz'
    baseline = float(c.get('reference_level', 10))
    threshold = float(c.get('plateau_fraction_threshold', 0.01))
    if not np.isfinite(baseline) or not np.isfinite(threshold) or not 0 < threshold <= 1:
        raise ValueError('Reference level must be finite and plateau threshold in (0, 1].')
    confirmed = c.get('conditions_confirmed', False) is True
    table = summary.copy()
    if 'role' in table:
        table = table[table['role'].eq('volume')]
    if 'window_variant' in table:
        table = table[table['window_variant'].eq('primary')]
    if 'volume_setting' not in table and not table.empty:
        raise ValueError('Volume summary needs volume_setting.')
    table['volume_setting'] = pd.to_numeric(table.get('volume_setting', pd.Series(dtype=float)), errors='coerce')
    rows = []
    for pair_index, pair in enumerate(pairs, 1):
        pair_id = f"{pair['im_file']} / {pair['accel_file']}"
        im = table[_vr_file_mask(table, pair['im_file'])]
        accel = table[_vr_file_mask(table, pair['accel_file'])]
        if 'sensor' in table:
            im = im[im['sensor'].astype(str).str.lower().isin(['im', 'inductive microphone'])]
            accel = accel[accel['sensor'].astype(str).str.lower().isin(['accel', 'accelerometer'])]
        im_file_count = im['path'].dropna().astype(str).nunique() if 'path' in im else int(not im.empty)
        accel_file_count = accel['path'].dropna().astype(str).nunique() if 'path' in accel else int(not accel.empty)
        levels = sorted(set(im['volume_setting'].dropna()) | set(accel['volume_setting'].dropna()))
        if not levels:
            levels = [np.nan]
        for level in levels:
            left = im[im['volume_setting'].eq(level)]
            right = accel[accel['volume_setting'].eq(level)]
            r = dict(pair_index=pair_index, pair_id=pair_id,
                     im_file=str(pair['im_file']), accel_file=str(pair['accel_file']),
                     volume_setting=level, reference_level=baseline,
                     conditions_confirmed=confirmed, amplitude_key=amplitude_key,
                     frequency_key=frequency_key, plateau_fraction_threshold=threshold,
                     im_row_count=len(left), accel_row_count=len(right),
                     im_matched_file_count=im_file_count, accel_matched_file_count=accel_file_count)
            reasons = []
            if not confirmed:
                reasons.append('conditions_not_confirmed')
            if im_file_count > 1:
                reasons.append('im_file_identity_ambiguous')
            if accel_file_count > 1:
                reasons.append('accel_file_identity_ambiguous')
            for prefix, part in (('im', left), ('accel', right)):
                if len(part) != 1:
                    reasons.append(prefix + ('_missing_volume' if len(part) == 0 else '_duplicate_volume'))
                source = part.iloc[0] if len(part) == 1 else pd.Series(dtype=object)
                r[prefix + '_amplitude'] = _vr_num(source, amplitude_key)
                r[prefix + '_frequency_hz'] = _vr_num(source, frequency_key)
                r[prefix + '_target_hz'] = _vr_num(source, 'target_hz')
                r[prefix + '_native_df_hz'] = _vr_num(source, 'native_df_hz')
                r[prefix + '_extrema_fraction'] = _vr_occupancy(source)
                r[prefix + '_unit'] = str(source.get('unit', 'recorded counts'))
                r[prefix + '_fft_computed'] = _vr_bool(source, 'fft_computed', _vr_bool(source, 'fft_valid'))
                r[prefix + '_timing_warning'] = bool(r[prefix + '_fft_computed'] and not _vr_bool(source, 'fft_valid'))
                r[prefix + '_quality_flags'] = str(source.get('quality_flags', ''))
                r[prefix + '_analysis_status'] = str(source.get('analysis_status', 'unavailable'))
                r[prefix + '_path'] = str(source.get('path', ''))
                for key in ('analysis_start_s', 'analysis_end_s', 'raw_rms', 'raw_peak_to_peak'):
                    r[prefix + '_' + key] = _vr_num(source, key)
                if not np.isfinite(r[prefix + '_amplitude']) or r[prefix + '_amplitude'] <= 0:
                    reasons.append(prefix + '_invalid_amplitude')
                if not r[prefix + '_fft_computed']:
                    reasons.append(prefix + '_fft_unavailable')
            ai, aa = r['im_amplitude'], r['accel_amplitude']
            r['ratio_unit'] = 'Data28 counts/g'
            r['ratio_raw_counts_per_g'] = ai / aa if np.isfinite(ai) and ai > 0 and np.isfinite(aa) and aa > 0 else np.nan
            ti, ta = r['im_target_hz'], r['accel_target_hz']
            nominal_match = np.isfinite(ti) and ti > 0 and np.isfinite(ta) and ta > 0 and np.isclose(ti, ta, rtol=0, atol=1e-9)
            r['target_frequencies_match'] = bool(nominal_match)
            if not nominal_match:
                reasons.append('target_frequency_mismatch_or_missing')
            di, da = r['im_native_df_hz'], r['accel_native_df_hz']
            tolerance = 2 * max(di, da) if np.isfinite(di) and di > 0 and np.isfinite(da) and da > 0 else np.nan
            r['frequency_compatibility_tolerance_hz'] = tolerance
            fi, fa = r['im_frequency_hz'], r['accel_frequency_hz']
            measured_match = np.isfinite(tolerance) and np.isfinite(fi) and np.isfinite(fa) and abs(fi - fa) <= tolerance
            target_supported = nominal_match and np.isfinite(tolerance) and np.isfinite(fi) and np.isfinite(fa) and abs(fi - ti) <= tolerance and abs(fa - ta) <= tolerance
            r['global_peaks_compatible'] = bool(measured_match)
            r['global_peaks_near_nominal'] = bool(target_supported)
            if not measured_match:
                reasons.append('global_peak_frequency_mismatch_or_missing')
            if not target_supported:
                reasons.append('global_peak_not_confirmed_near_nominal')
            occupancy = r['accel_extrema_fraction']
            r['accel_plateau_flag'] = bool(np.isfinite(occupancy) and occupancy >= threshold)
            if not np.isfinite(occupancy) or not 0 <= occupancy <= 1:
                reasons.append('accelerometer_extrema_occupancy_unknown_or_invalid')
            elif r['accel_plateau_flag']:
                reasons.append('accelerometer_repeated_extrema')
            r['timing_warning'] = r['im_timing_warning'] or r['accel_timing_warning']
            r['ratio_usable'] = not reasons
            r['ratio_usable_counts_per_g'] = r['ratio_raw_counts_per_g'] if r['ratio_usable'] else np.nan
            r['comparison_status'] = ('unavailable_or_flagged' if reasons else
                                      'diagnostic_timing_warning' if r['timing_warning'] else
                                      'conditional_comparison')
            r['comparison_reasons'] = ';'.join(reasons)
            r['ratio_definition'] = VOLUME_REFERENCE_METHODS['ratio']
            r['ratio_usable_definition'] = VOLUME_REFERENCE_METHODS['ratio_usable']
            rows.append(r)
    result = pd.DataFrame(rows)
    for key in ('im_normalized', 'accel_normalized', 'im_reference_amplitude', 'accel_reference_amplitude'):
        result[key] = np.nan
    result['im_normalization_status'] = 'missing_baseline'
    result['accel_normalization_status'] = 'missing_baseline'
    for _, group in result.groupby('pair_index', sort=False):
        baseline_rows = group[group['volume_setting'].eq(baseline)]
        for sensor in ('im', 'accel'):
            baseline_value = np.nan
            status = 'missing_baseline'
            if len(baseline_rows) == 1:
                b = baseline_rows.iloc[0]
                value = b[sensor + '_amplitude']
                if b[sensor + '_row_count'] != 1 or b[sensor + '_matched_file_count'] > 1:
                    status = 'ambiguous_baseline'
                elif not np.isfinite(value) or value <= 0 or not b[sensor + '_fft_computed']:
                    status = 'invalid_baseline_amplitude'
                elif sensor == 'accel' and (not np.isfinite(b['accel_extrema_fraction']) or b['accel_plateau_flag']):
                    status = 'flagged_or_unknown_accelerometer_baseline'
                else:
                    baseline_value = value
                    status = 'diagnostic_timing_warning' if b[sensor + '_timing_warning'] else 'defined'
            result.loc[group.index, sensor + '_reference_amplitude'] = baseline_value
            result.loc[group.index, sensor + '_normalization_status'] = status
            values = group[sensor + '_amplitude'].where(group[sensor + '_amplitude'].gt(0))
            values = values.where(group[sensor + '_fft_computed'])
            if np.isfinite(baseline_value):
                result.loc[group.index, sensor + '_normalized'] = values / baseline_value
    result.attrs['methods'] = dict(VOLUME_REFERENCE_METHODS)
    result.attrs['effective_config'] = dict(pairs=pairs, conditions_confirmed=confirmed,
        reference_level=baseline, amplitude_key=amplitude_key,
        plateau_fraction_threshold=threshold, peak_match_native_bins=2)
    return result


def _vr_style(ax, xlabel, ylabel, title):
    ax.set(xlabel=xlabel, ylabel=ylabel, title=title)
    ax.grid(True, alpha=0.25)
    ax.spines[['top', 'right']].set_visible(False)
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        unique = dict(zip(labels, handles))
        ax.legend(unique.values(), unique.keys(), fontsize=8, loc='best')


def _vr_mark_flags(ax, x, y, plateau, timing):
    x, y = np.asarray(x, float), np.asarray(y, float)
    valid = np.isfinite(x) & np.isfinite(y)
    plateau, timing = np.asarray(plateau, bool) & valid, np.asarray(timing, bool) & valid
    if plateau.any():
        ax.scatter(x[plateau], y[plateau], marker='s', facecolors='none', edgecolors='firebrick',
                   s=105, linewidths=1.4, zorder=5, label='Accelerometer repeated extrema')
    if timing.any():
        ax.scatter(x[timing], y[timing], marker='o', facecolors='none', edgecolors='darkorange',
                   s=66, linewidths=1.2, zorder=6, label='Timestamp warning')


def plot_volume(summary, title, subwindows=None, plateau_fraction_threshold=0.01):
    """Three distinct amplitudes versus ascending numeric volume; subwindows are not repeats."""
    threshold = float(plateau_fraction_threshold)
    if not np.isfinite(threshold) or not 0 < threshold <= 1:
        raise ValueError('plateau_fraction_threshold must be in (0, 1].')
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), layout='constrained')
    fig.suptitle(title)
    if summary.empty or 'volume_setting' not in summary:
        axes[0].text(0.5, 0.5, 'No volume windows available.', ha='center', transform=axes[0].transAxes)
        return fig
    table = summary.copy()
    table['volume_setting'] = pd.to_numeric(table['volume_setting'], errors='coerce')
    table = table.sort_values('volume_setting')
    units = table.get('unit', pd.Series('counts', index=table.index)).dropna().astype(str).unique()
    unit = units[0] if len(units) == 1 else 'recorded units (see table)'
    columns = [('global_padded_parabolic_amplitude', 'Global FFT peak: padded + quadratic fit', 'Peak amplitude'),
               ('raw_rms', 'Demeaned raw RMS: all components', 'RMS amplitude'),
               ('raw_peak_to_peak', 'Raw peak-to-peak: extrema', 'Peak-to-peak amplitude')]
    grouping = 'path' if 'path' in table else 'file_label' if 'file_label' in table else None
    groups = list(table.groupby(grouping, sort=False, dropna=False)) if grouping else [('record', table)]
    for group_index, (file_id, group) in enumerate(groups):
        x = group['volume_setting'].to_numpy(float)
        duration = pd.to_numeric(group.get('analysis_end_s', pd.Series(np.nan, index=group.index)), errors='coerce') - pd.to_numeric(group.get('analysis_start_s', pd.Series(np.nan, index=group.index)), errors='coerce')
        label = 'Central 20 s' if np.all(np.isfinite(duration)) and np.allclose(duration, 20) else 'Primary window'
        if len(groups) > 1:
            label += ': ' + Path(str(file_id)).stem
        color = f'C{group_index % 10}'
        computed = group.get('fft_computed', group.get('fft_valid', pd.Series(False, index=group.index))).fillna(False).astype(bool)
        timing = computed & ~group.get('fft_valid', pd.Series(False, index=group.index)).fillna(False).astype(bool)
        sensors = group.get('sensor', pd.Series('', index=group.index)).astype(str).str.lower()
        occupancy = group.apply(_vr_occupancy, axis=1)
        plateau = sensors.isin(['accel', 'accelerometer']) & occupancy.ge(threshold)
        for ax, (key, panel_title, ylabel) in zip(axes, columns):
            y = pd.to_numeric(group.get(key, pd.Series(np.nan, index=group.index)), errors='coerce').to_numpy(float)
            # Duplicate volume conditions are individual observations, never an implicit mean.
            line = 'none' if group['volume_setting'].duplicated().any() else '-'
            ax.plot(x, y, marker='o', linestyle=line, markersize=5, color=color, label=label)
            _vr_mark_flags(ax, x, y, plateau, timing if key.endswith('_amplitude') else np.zeros(len(group), bool))
            if subwindows is not None and not subwindows.empty and key in subwindows:
                sub = subwindows.copy()
                if grouping and grouping in sub:
                    sub = sub[sub[grouping].eq(file_id)]
                if 'volume_setting' in sub:
                    sub_label = 'Subwindows (not repeats)'
                    if {'analysis_start_s', 'analysis_end_s'} <= set(sub.columns) and not sub.empty:
                        sub_duration = (pd.to_numeric(sub['analysis_end_s'], errors='coerce')
                                        - pd.to_numeric(sub['analysis_start_s'], errors='coerce')).to_numpy(float)
                        if np.isfinite(sub_duration).all() and np.all(sub_duration > 0):
                            if np.allclose(sub_duration, sub_duration[0], rtol=0, atol=1e-9):
                                sub_label = f'{sub_duration[0]:g} s subwindows (not repeats)'
                            else:
                                sub_label = f'{sub_duration.min():g}?{sub_duration.max():g} s subwindows (not repeats)'
                    ax.scatter(pd.to_numeric(sub['volume_setting'], errors='coerce'),
                               pd.to_numeric(sub[key], errors='coerce'), marker='x', s=35,
                               color=color, alpha=0.5, label=sub_label)
            _vr_style(ax, 'Amplifier volume setting (ordinal)', f'{ylabel} [{unit}]', panel_title)
    levels = sorted(table['volume_setting'].dropna().unique())
    for ax in axes:
        ax.set_xticks(levels)
    fig.supxlabel('Separate amplitude definitions; original 15-to-10 acquisition is displayed from 10 to 15. '
                  'Subwindows show within-record variation; volume is not calibrated excitation.', fontsize=9)
    return fig


def plot_volume_reference(table):
    """Normalized trends, labelled condition scatter, and conditional counts/g ratio; no fit."""
    figures = []
    if table.empty:
        fig, ax = plt.subplots(figsize=(10, 4), layout='constrained')
        ax.text(0.5, 0.5, 'No configured volume reference pairs available.', ha='center', transform=ax.transAxes)
        return [fig]
    for _, group in table.groupby('pair_index', sort=False):
        group = group.sort_values('volume_setting')
        fig, axes = plt.subplots(1, 3, figsize=(18, 5.6), layout='constrained')
        pair, baseline = group['pair_id'].iloc[0], group['reference_level'].iloc[0]
        fig.suptitle(f'{pair}: matched volume conditions, sequential recordings')
        x = group['volume_setting'].to_numpy(float)
        plateau = group['accel_plateau_flag'].to_numpy(bool)
        timing = group['timing_warning'].to_numpy(bool)
        for sensor, label, marker in (('im', 'IM', 'o'), ('accel', 'Accelerometer', '^')):
            values = group[sensor + '_normalized'].to_numpy(float)
            axes[0].plot(x, values, marker=marker, markersize=5, linewidth=1, label=label)
            _vr_mark_flags(axes[0], x, values, plateau if sensor == 'accel' else np.zeros(len(group), bool),
                           group[sensor + '_timing_warning'].to_numpy(bool))
            if not np.isfinite(values).any():
                text = ', '.join(group[sensor + '_normalization_status'].unique())
                axes[0].text(0.02, 0.96 if sensor == 'im' else 0.83,
                    f'{label} normalization unavailable:\n{text}', va='top', fontsize=8, transform=axes[0].transAxes)
        axes[0].axhline(1, color='0.6', linestyle=':', linewidth=1)
        _vr_style(axes[0], 'Amplifier volume setting (ordinal)', f'Amplitude / own volume {baseline:g} amplitude',
                  'Relative amplitude trends')
        xa, yi = group['accel_amplitude'].to_numpy(float), group['im_amplitude'].to_numpy(float)
        axes[1].scatter(xa, yi, color='C0', s=40, label='Matched volume condition')
        _vr_mark_flags(axes[1], xa, yi, plateau, timing)
        for level, a, i in zip(x, xa, yi):
            if np.isfinite(level) and np.isfinite(a) and np.isfinite(i):
                axes[1].annotate(f'Vol {level:g}', (a, i), xytext=(4, 4), textcoords='offset points', fontsize=8)
        _vr_style(axes[1], 'Accelerometer peak amplitude [g]', 'IM peak amplitude [Data28 counts]',
                  'Separate recordings at the same volume')
        raw = group['ratio_raw_counts_per_g'].to_numpy(float)
        usable = group['ratio_usable'].to_numpy(bool)
        if (~usable & np.isfinite(raw)).any():
            axes[2].scatter(x[~usable], raw[~usable], color='0.55', marker='x', s=50,
                            label='Flagged descriptive ratio')
        axes[2].plot(x, group['ratio_usable_counts_per_g'], 'o-', color='C0', markersize=5,
                     label='Conditional comparison')
        _vr_mark_flags(axes[2], x, raw, plateau, timing)
        for xx, yy, allowed in zip(x, raw, usable):
            if np.isfinite(xx) and np.isfinite(yy) and not allowed:
                axes[2].annotate('flagged', (xx, yy), xytext=(3, 6), textcoords='offset points', fontsize=7, color='0.45')
        _vr_style(axes[2], 'Amplifier volume setting (ordinal)', 'IM peak / accelerometer peak [Data28 counts/g]',
                  'Conditional amplitude-ratio proxy')
        for ax in (axes[0], axes[2]):
            ax.set_xticks(x[np.isfinite(x)])
        nusable, nflagged = int(usable.sum()), int(plateau.sum())
        fig.supxlabel(f'{nusable}/{len(group)} numerical comparisons pass the selected condition/peak/extrema checks; '
                      f'{nflagged} accelerometer repeated-extrema flags.\n'
                      'Timestamp-warning points remain diagnostic. Separate acquisitions may drift; counts/g ratios '
                      'are not calibrated sensitivity. No regression or synchronized transfer function.', fontsize=9)
        figures.append(fig)
    return figures


In [ ]:
# sinewave-cell: resolution_statistics
from resolution_frequency_comparison import (
    RESOLUTION_METHODS, COMPARISON_METHODS, sine_fit_settings, add_frequency_estimators,
    frequency_estimator_table, plot_estimator_errors, plot_sine_fit_agreement,
)
from resolution_relative_summary import (
    RELATIVE_SUMMARY_METHODS, build_relative_summary, plot_relative_frequency, plot_absolute_ppm,
)
ESTIMATORS = [name for name, _ in RESOLUTION_METHODS]


def summarize_resolution(table, estimate_view="computed"):
    if estimate_view not in {"computed", "strict_gap"}:
        raise ValueError("Unknown estimate_view")
    prefix = "strict_" if estimate_view == "strict_gap" else ""
    if table.empty:
        return pd.DataFrame(), pd.DataFrame()
    groups = []
    for key, part in table.groupby(["base_hz", "target_hz", "window_variant"], sort=True):
        base, target, variant = key
        if part["path"].duplicated().any():
            raise ValueError("A file cannot count twice in one resolution group")
        for method in ESTIMATORS:
            values = pd.to_numeric(part[prefix + method + "_hz"], errors="coerce")
            values = values[np.isfinite(values)]
            n = len(values)
            groups.append({"base_hz": base, "target_hz": target, "window_variant": variant,
                "method": method, "estimate_view": estimate_view, "n_files": len(part), "n_valid": n,
                "n_strict_gap_pass": int(np.isfinite(part["strict_" + method + "_hz"]).sum()),
                "mean_hz": values.mean() if n else np.nan,
                "mean_error_hz": values.mean() - target if n else np.nan,
                "sample_sd_hz": values.std(ddof=1) if n >= 2 else np.nan,
                "min_hz": values.min() if n else np.nan, "max_hz": values.max() if n else np.nan,
                "observed_range_hz": values.max() - values.min() if n else np.nan,
                "quality_flagged_records": int(part["quality_flags"].fillna("").ne("").sum()),
                "repeatability_status": "sample_sd_available" if n >= 2 else
                    ("single_record_no_repeatability_estimate" if n == 1 else "no_valid_estimate"),
                "aggregation": "unweighted record-level arithmetic mean of finite estimates in specified view; computed view includes timing-warning diagnostics"})
    grouped = pd.DataFrame(groups)
    comparisons = []
    for (base, variant, method), part in grouped.groupby(["base_hz", "window_variant", "method"], sort=True):
        for low, high in combinations(part.sort_values("target_hz").to_dict("records"), 2):
            nominal = high["target_hz"] - low["target_hz"]
            measured = high["mean_hz"] - low["mean_hz"]
            comparisons.append({"base_hz": base, "window_variant": variant, "method": method,
                "lower_target_hz": low["target_hz"], "upper_target_hz": high["target_hz"],
                "estimate_view": estimate_view, "nominal_difference_hz": nominal, "measured_difference_of_means_hz": measured,
                "difference_error_hz": measured - nominal,
                "lower_n_valid": low["n_valid"], "upper_n_valid": high["n_valid"],
                "lower_sample_sd_hz": low["sample_sd_hz"], "upper_sample_sd_hz": high["sample_sd_hz"],
                "observed_interval_gap_hz": high["min_hz"] - low["max_hz"],
                "interpretation": "descriptive independent-record difference; no calibrated uncertainty or two-tone resolution claim"})
    return grouped, pd.DataFrame(comparisons)


In [ ]:
# sinewave-cell: snr_helpers
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import welch
from scipy.signal.windows import hann


SNR_METHODS = {
    "metric": "Nominal-band excess-power SNR against each separate noise recording; supplemental to global FFT peak estimates, which are unchanged.",
    "signal_estimator": "Integrate the native one-sided periodic-Hann periodogram PSD of each excited analysis window over the configured nominal-frequency band. Estimated excess power = measured excited band power minus reference noise band power. This is band excess, not an isolated fundamental estimator.",
    "noise_estimator": "Welch PSD on each explicitly configured same-sensor control separately: default 6 s periodic Hann, 50% overlap, per-segment constant detrend, nfft=nperseg, arithmetic mean in linear PSD. Requested nperseg is never silently shortened. No averaging between control recordings.",
    "preprocessing": "Use the existing fixed-N linear timestamp interpolation and fs=(N-1)/retained span. Excited periodogram removes its window mean; Welch removes each segment mean. No additional bandpass or other filter, accel uses the same nominal counts-to-g conversion for signal and noise when enabled; no zero padding for PSD integration.",
    "bandwidth": "Default nominal target +/-1 Hz, clipped to common represented PSD frequency support. Integrate piecewise-linear PSD using trapezoids with interpolated values at identical physical band edges for signal and noise. Retain the DC endpoint after detrending; no original DC level is added. Common support ends at min(last positive excited bin,last noise bin), at or just below Nyquist. Report exact edges, width, df, ENBW, and clipping.",
    "linear_and_db": "SNR_power=(P_excited-P_noise)/P_noise; SNR_dB=10*log10(SNR_power) only for positive finite noise and positive excess. Nonpositive excess remains recorded, while its SNR_dB and SNR_power are NaN, never floored. Independently retain measured band-power ratio P_excited/P_noise and 10*log10 of that ratio; zero measured power gives -inf, not an invented floor.",
    "assumptions": "Subtraction assumes the separate control represents the background during excitation and background/excitation powers add without coherent cross terms. These are unverified conditions, not synchronization. IM_1 nothing-on comparison includes any amplifier-added power; IM_2 amplifier-on is a separate reference. Other components and fixed lines inside the band contribute, including interference around 200 Hz. Results do not establish fundamental detection, detection limit, calibrated sensitivity, or physical frequency response.",
    "aggregation_and_quality": "One value per excited window and each configured control; no averaging of targets or reference recordings, no confidence interval from overlapping Welch segments. Retain both records' actual windows, time-gap eligibility, flags and PSD settings. Unavailable, ambiguous, wrong-sensor and undersupported references stay flagged with missing values. Computational availability does not imply calibrated validity.",
}


def _snr_settings(config):
    cfg = dict(enabled=True, sensors=["IM"], noise_files={sensor: [Path(path).stem for path in controls.values()]
                            for sensor, controls in FILES["noise"].items()},
               band_halfwidth_hz=1.0, noise_segment_s=6.0, noise_overlap=0.5)
    if config:
        supplied = config.get("snr", config)
        unknown = set(supplied) - set(cfg)
        if unknown:
            raise ValueError(f"Unknown SNR settings: {sorted(unknown)}")
        cfg.update(supplied)
    for key in ("band_halfwidth_hz", "noise_segment_s"):
        if not np.isfinite(cfg[key]) or cfg[key] <= 0:
            raise ValueError(f"SNR {key} must be finite and positive.")
    if not np.isfinite(cfg["noise_overlap"]) or not 0 <= cfg["noise_overlap"] < 1:
        raise ValueError("SNR noise_overlap must be in [0, 1).")
    for sensor, names in cfg["noise_files"].items():
        if not isinstance(names, (list, tuple)) or len(set(names)) != len(names):
            raise ValueError(f"SNR noise_files[{sensor}] must list distinct explicit file IDs.")
    return cfg


def _snr_true(value):
    return value is not None and not pd.isna(value) and bool(value)


def _snr_finite(value):
    try:
        return bool(np.isfinite(float(value)))
    except (TypeError, ValueError):
        return False


def _snr_file(row):
    return str(row.get("file_id", row.get("file_label", "")))


def prepare_snr_noise(row, spectrum, config=None):
    """Return one explicit control's Welch PSD and effective metadata; never read data."""
    cfg = _snr_settings(config)
    ref = dict(row=dict(row), f=np.array([], dtype=float), psd=np.array([], dtype=float),
               status="unavailable", file_id=_snr_file(row), sensor=row.get("sensor", ""),
               noise_segment_s_requested=float(cfg["noise_segment_s"]),
               noise_overlap_fraction_requested=float(cfg["noise_overlap"]),
               noise_nperseg=0, noise_noverlap=0, noise_nfft=0,
               noise_welch_segment_count=0, noise_df_hz=np.nan, noise_enbw_hz=np.nan,
               noise_segment_duration_s=np.nan, noise_overlap_fraction_actual=np.nan,
               noise_welch_used_sample_count=0, noise_welch_unused_tail_samples=0,
               noise_welch_first_sample_s=np.nan, noise_welch_last_sample_s=np.nan,
               noise_psd_aggregation="Welch arithmetic mean in linear power; separate control",
               noise_psd_window="periodic Hann", noise_psd_detrend="constant per segment",
               noise_psd_scaling="one-sided density, input-count^2/Hz",
               noise_additional_filter="none", noise_zero_padding=False)
    if not cfg["enabled"]:
        ref["status"] = "disabled"
        return ref
    if not _snr_true(row.get("fft_computed", False)):
        ref["status"] = "noise_fft_unavailable"
        return ref
    fs = row.get("fs_hz", np.nan)
    y = np.asarray(spectrum.get("uniform_y", []), dtype=float)
    if not _snr_finite(fs) or fs <= 0 or y.ndim != 1 or not np.all(np.isfinite(y)):
        ref["status"] = "noise_uniform_samples_or_fs_invalid"
        return ref
    nperseg = int(round(cfg["noise_segment_s"] * fs))
    if nperseg < 4:
        ref["status"] = "noise_segment_fewer_than_four_samples"
        return ref
    noverlap = int(np.floor(cfg["noise_overlap"] * nperseg))
    ref.update(noise_nperseg=nperseg, noise_noverlap=noverlap, noise_nfft=nperseg,
               noise_df_hz=float(fs / nperseg), noise_segment_duration_s=float(nperseg / fs),
               noise_overlap_fraction_actual=float(noverlap / nperseg))
    if len(y) < nperseg:
        ref["status"] = "noise_shorter_than_requested_welch_segment"
        return ref
    window = hann(nperseg, sym=False)
    f, psd = welch(y, fs=fs, window=window, nperseg=nperseg, noverlap=noverlap,
                   nfft=nperseg, detrend="constant", return_onesided=True,
                   scaling="density", average="mean")
    count = 1 + (len(y) - nperseg) // (nperseg - noverlap)
    used = nperseg + (count - 1) * (nperseg - noverlap)
    start = float(row.get("actual_start_s", np.nan))
    ref.update(status="computed", f=f, psd=psd, noise_welch_segment_count=int(count),
               noise_enbw_hz=float(fs * np.sum(window**2) / window.sum()**2),
               noise_welch_used_sample_count=int(used),
               noise_welch_unused_tail_samples=int(len(y) - used),
               noise_welch_first_sample_s=start,
               noise_welch_last_sample_s=float(start + (used - 1) / fs))
    return ref


def _snr_spectrum_ok(f, psd):
    return (f.ndim == 1 and psd.ndim == 1 and len(f) == len(psd) and len(f) >= 2
            and np.all(np.isfinite(f)) and np.all(np.diff(f) > 0)
            and np.all(np.isfinite(psd)) and np.all(psd >= 0))


def _snr_band_power(f, psd, low, high):
    """Exact integral of the piecewise-linear sampled PSD on common physical edges."""
    inside = (f > low) & (f < high)
    x = np.concatenate(([low], f[inside], [high]))
    y = np.interp(x, f, psd)
    return float(np.sum(0.5 * (y[1:] + y[:-1]) * np.diff(x)))


def build_sweep_snr(sweep_items, noise_references, config=None):
    """Compare (row, native spectrum) items against explicit prepared controls only.

    noise_references is a list of prepare_snr_noise return values. Missing and
    ambiguous IDs produce rows with unavailable status rather than implicit pairing.
    """
    cfg = _snr_settings(config)
    if not cfg["enabled"]:
        return pd.DataFrame()
    references = list(noise_references)
    rows = []
    retain = ("analysis_start_s", "analysis_end_s", "actual_start_s", "actual_end_s",
              "requested_duration_s", "sample_count", "fs_hz", "nyquist_hz",
              "native_df_hz", "enbw_hz", "fft_computed", "fft_valid", "analysis_status",
              "strict_gap_check_passed", "strict_rejection_flags", "quality_flags",
              "gap_interval_count", "gap_duration_fraction", "max_gap_over_median",
              "timestamp_resampling", "preprocessing", "unit")
    for source, spectrum in sweep_items:
        sensor = source.get("sensor", "")
        if sensor not in cfg["sensors"]:
            continue
        for noise_file in cfg["noise_files"].get(sensor, []):
            r = dict(sensor=sensor, file_label=_snr_file(source), excited_file=_snr_file(source),
                     path=source.get("path", ""), excited_path=source.get("path", ""),
                     noise_file=str(noise_file), noise_path="",
                     noise_condition="", target_hz=source.get("target_hz", np.nan),
                     segment_index=source.get("segment_index", np.nan),
                     availability_status="unavailable", snr_status="unavailable",
                     requested_band_halfwidth_hz=float(cfg["band_halfwidth_hz"]),
                     requested_band_low_hz=np.nan, requested_band_high_hz=np.nan,
                     band_low_hz=np.nan, band_high_hz=np.nan, effective_bandwidth_hz=np.nan,
                     band_clipped_low=False, band_clipped_high=False,
                     excited_native_bins_in_band=0, noise_native_bins_in_band=0,
                     excited_band_power=np.nan, noise_band_power=np.nan,
                     excess_band_power=np.nan, snr_power_linear=np.nan, snr_db=np.nan,
                     measured_power_ratio=np.nan, measured_power_ratio_db=np.nan,
                     global_peak_hz=source.get("global_bin_hz", np.nan),
                     global_peak_outside_band=False, excited_timing_warning=False,
                     noise_timing_warning=False, either_timing_warning=False,
                     both_strict_gap_checks_passed=False, quality_flags="",
                     power_unit=str(source.get("unit", "counts")) + " squared",
                     signal_estimator="native one-sided Hann PSD integral minus separate noise-band power",
                     band_integration="piecewise-linear trapezoidal integral with interpolated common edges; detrended DC endpoint retained",
                     snr_definition="(P_excited-P_noise)/P_noise; 10*log10 for positive excess only",
                     measured_ratio_definition="P_excited/P_noise; 10*log10, includes background",
                     interpretation="nominal-band excess-power SNR; separate noise reference; not isolated fundamental or detection limit")
            for key in retain:
                r["excited_" + key] = source.get(key, np.nan)
                r["noise_" + ("record_enbw_hz" if key == "enbw_hz" else key)] = np.nan
            flags = []
            r["excited_timing_warning"] = (_snr_true(source.get("fft_computed", False))
                                              and not _snr_true(source.get("strict_gap_check_passed", False)))
            r["either_timing_warning"] = r["excited_timing_warning"]
            matches = [ref for ref in references if ref.get("sensor") == sensor
                       and ref.get("file_id") == str(noise_file)]
            if len(matches) != 1:
                r["snr_status"] = "noise_reference_missing" if not matches else "noise_reference_ambiguous"
                r["quality_flags"] = r["snr_status"]
                rows.append(r)
                continue
            ref = matches[0]
            noise = ref["row"]
            r.update({key: value for key, value in ref.items() if key.startswith("noise_")})
            r.update(noise_path=noise.get("path", ""), noise_condition=noise.get("noise_condition", ""),
                     noise_reference_status=ref.get("status", "unavailable"))
            for key in retain:
                r["noise_" + ("record_enbw_hz" if key == "enbw_hz" else key)] = noise.get(key, np.nan)
            r["noise_timing_warning"] = (_snr_true(noise.get("fft_computed", False))
                                            and not _snr_true(noise.get("strict_gap_check_passed", False)))
            r["either_timing_warning"] = r["excited_timing_warning"] or r["noise_timing_warning"]
            r["both_strict_gap_checks_passed"] = (_snr_true(source.get("strict_gap_check_passed", False))
                                                  and _snr_true(noise.get("strict_gap_check_passed", False)))
            if r["excited_timing_warning"]:
                flags.append("excited_internal_gap_interpolated")
            if r["noise_timing_warning"]:
                flags.append("noise_internal_gap_interpolated")
            if source.get("quality_flags", ""):
                flags.append("excited_quality_flags_present")
            if noise.get("quality_flags", ""):
                flags.append("noise_quality_flags_present")
            if not _snr_true(source.get("fft_computed", False)):
                r["snr_status"] = "excited_fft_unavailable"
            elif ref.get("status") != "computed":
                r["snr_status"] = ref.get("status", "noise_unavailable")
            elif source.get("unit", "counts") != noise.get("unit", "counts"):
                r["snr_status"] = "signal_noise_units_differ"
            elif not _snr_finite(r["target_hz"]) or r["target_hz"] <= 0:
                r["snr_status"] = "invalid_nominal_target"
            else:
                sf = np.asarray(spectrum.get("f", []), dtype=float)
                sp = np.asarray(spectrum.get("psd", []), dtype=float)
                nf, npower = np.asarray(ref["f"], dtype=float), np.asarray(ref["psd"], dtype=float)
                if not _snr_spectrum_ok(sf, sp) or not _snr_spectrum_ok(nf, npower):
                    r["snr_status"] = "invalid_or_unavailable_psd"
                else:
                    low_req = float(r["target_hz"] - cfg["band_halfwidth_hz"])
                    high_req = float(r["target_hz"] + cfg["band_halfwidth_hz"])
                    low = float(max(low_req, 0.0, sf[0], nf[0]))
                    high = float(min(high_req, sf[-1], nf[-1]))
                    r.update(requested_band_low_hz=low_req, requested_band_high_hz=high_req,
                             band_low_hz=low, band_high_hz=high,
                             effective_bandwidth_hz=max(0.0, high-low),
                             band_clipped_low=bool(low > low_req + 1e-12),
                             band_clipped_high=bool(high < high_req - 1e-12))
                    if low >= high or not low <= r["target_hz"] <= high:
                        r["snr_status"] = "nominal_band_or_target_outside_common_support"
                    else:
                        pe = _snr_band_power(sf, sp, low, high)
                        pn = _snr_band_power(nf, npower, low, high)
                        r.update(availability_status="computed", excited_band_power=pe,
                                 noise_band_power=pn, excess_band_power=pe-pn,
                                 excited_native_bins_in_band=int(np.count_nonzero((sf >= low) & (sf <= high))),
                                 noise_native_bins_in_band=int(np.count_nonzero((nf >= low) & (nf <= high))))
                        if r["band_clipped_low"] or r["band_clipped_high"]:
                            flags.append("nominal_band_clipped_to_common_psd_support")
                        if _snr_finite(r["global_peak_hz"]) and not low <= r["global_peak_hz"] <= high:
                            r["global_peak_outside_band"] = True
                            flags.append("global_fft_max_outside_nominal_band")
                        if min(r["excited_native_bins_in_band"], r["noise_native_bins_in_band"]) < 3:
                            flags.append("fewer_than_three_native_psd_bins_in_band")
                        enbws = [source.get("enbw_hz", np.nan), ref.get("noise_enbw_hz", np.nan)]
                        if any(_snr_finite(v) and high-low < v for v in enbws):
                            flags.append("band_narrower_than_hann_enbw")
                        if not np.isfinite(pe) or not np.isfinite(pn):
                            r["snr_status"] = "nonfinite_integrated_power"
                        elif pn <= 0:
                            r["snr_status"] = "nonpositive_reference_noise_power"
                        else:
                            ratio = pe / pn
                            r.update(measured_power_ratio=ratio,
                                     measured_power_ratio_db=float(10*np.log10(ratio)) if ratio > 0 else -np.inf)
                            if pe > pn:
                                excess_ratio = (pe-pn) / pn
                                r.update(snr_status="positive_excess", snr_power_linear=excess_ratio,
                                         snr_db=float(10*np.log10(excess_ratio)))
                            else:
                                r["snr_status"] = "nonpositive_excess"
            if r["snr_status"] != "positive_excess":
                flags.append(r["snr_status"])
            r["quality_flags"] = ";".join(flags)
            rows.append(r)
    return pd.DataFrame(rows)


def plot_sweep_snr(table, title="Frequency sweep: separate-noise-reference comparison"):
    """Show excess-power SNR and distinct measured ratio; undefined SNR stays missing."""
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    fig.suptitle(title)
    axes[0].set_title("Nominal-band excess-power SNR")
    axes[1].set_title("Measured excited / noise band-power ratio")
    axes[0].set_ylabel("10 log10[(Pexcited - Pnoise) / Pnoise] [dB]")
    axes[1].set_ylabel("10 log10[Pexcited / Pnoise] [dB]")
    for ax in axes:
        ax.set_xscale("log")
        ax.set_xlabel("Nominal sine frequency [Hz]")
        ax.axhline(0, color="0.5", lw=0.8)
        ax.grid(True, which="both", alpha=0.22)
    if table.empty:
        for ax in axes:
            ax.text(0.5, 0.5, "No configured sweep / noise comparisons", ha="center",
                    va="center", transform=ax.transAxes)
        fig.tight_layout()
        return fig
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    undefined_count, timing_count = 0, 0
    groups = table.groupby(["excited_file", "noise_file"], sort=False, dropna=False)
    multiple_excited = table["excited_file"].nunique() > 1
    for index, ((excited_file, noise_file), group) in enumerate(groups):
        group = group.sort_values("target_hz")
        x = pd.to_numeric(group["target_hz"], errors="coerce").to_numpy(dtype=float)
        keep = np.isfinite(x) & (x > 0)
        group, x = group.loc[keep], x[keep]
        if not len(group):
            continue
        color = colors[index % len(colors)]
        condition = str(group["noise_condition"].iloc[0])
        label = f"{noise_file}: {condition}" if condition and condition != "nan" else str(noise_file)
        if multiple_excited:
            label = str(excited_file) + " vs " + label
        for ax, key in zip(axes, ("snr_db", "measured_power_ratio_db")):
            y = pd.to_numeric(group[key], errors="coerce").to_numpy(dtype=float)
            plotted_y = np.where(np.isfinite(y), y, np.nan)
            ax.plot(x, plotted_y, "o-", ms=4, color=color, label=label)
            warning = group["either_timing_warning"].fillna(False).to_numpy(dtype=bool) & np.isfinite(y)
            if warning.any():
                ax.scatter(x[warning], y[warning], s=65, facecolors="none", edgecolors="darkorange",
                           linewidths=1.2, zorder=4)
        undefined = ~np.isfinite(pd.to_numeric(group["snr_db"], errors="coerce").to_numpy(dtype=float))
        if undefined.any():
            axes[0].scatter(x[undefined], np.full(undefined.sum(), 0.025+0.025*index),
                            transform=axes[0].get_xaxis_transform(), color=color, marker="x", s=35,
                            clip_on=False, label=f"{noise_file}: undefined SNR ({undefined.sum()})")
        undefined_count += int(undefined.sum())
        timing_count += int(group["either_timing_warning"].fillna(False).sum())
    for ax in axes:
        ax.legend(fontsize=8, loc="best")
    widths = pd.to_numeric(table["requested_band_halfwidth_hz"], errors="coerce").dropna().unique()
    band_text = f"nominal +/- {widths[0]:g} Hz" if len(widths) == 1 else "configured nominal band"
    fig.text(0.5, 0.035,
             f"Band: {band_text}, clipped to common support; actual bandwidths in CSV. "
             f"Orange rings: {timing_count} comparisons with timestamp-gap warnings.\n"
             f"Bottom crosses: {undefined_count} undefined excess SNR values (not plotted at an artificial dB floor). "
             "Separate controls; background matching/additive power assumed; in-band interference is included.",
             ha="center", va="bottom", fontsize=8)
    fig.tight_layout(rect=(0, 0.12, 1, 0.94))
    return fig


In [ ]:
# sinewave-cell: run_helpers
METHODS = {
    "experiment": "Prior experimental context reports a sine generator setting of 1 Vpp; photos are setup references. For added IM_39-45, only the nominal frequency/file mapping is confirmed; generator amplitude and other acquisition conditions are unverified. No measured transducer input or acoustic reference.",
    "units": "Time_us * 1e-6 relative to first timestamp; IM Data28 counts; accel Raw_Z converted to g using each recording full scale and nominal ASM330LHH sensitivity (2/4/8/16 g: 0.000061/0.000122/0.000244/0.000488 g/count); no individual calibration. Original CSV unchanged; raw_counts retained in loaded record.",
    "timing": "Half-open [start,end) windows. Nominal schedules begin at configured start_s, not inferred switches. Explicit bounds override schedule; trim follows bounds. Frequency selection is applied AFTER complete schedule construction.",
    "support": "Requested window must be within recording; at record end allow exactly one full-record mean sample interval plus floating tolerance for conventional half-open support. Actual first/last samples and use of tolerance reported; no silent shortening.",
    "sampling": "Reject nonfinite/duplicate/decreasing timestamps. Estimate fs=(N-1)/(last-first) per retained window. Interpolate linearly to N equispaced times; no clock correction from targets.",
    "fft": "Remove uniform-signal mean; periodic Hann; no additional filters. Global argmax over every positive rFFT bin through Nyquist when present; exclude DC only. Targets never restrict search.",
    "amplitude": "One-sided sinusoidal peak amplitude 2*abs(FFT)/sum(window), except DC and exact Nyquist not doubled. Native/padded transform lengths differ; normalization always original window sum. Dominant peak need not be excitation fundamental.",
    "interpolation": "Configured interpolation_points actual adjacent bins around global maximum (default 3, optional 5; both counts additionally retained in the explicit seven-method comparison); unweighted degree-2 least-squares fit in centered bin offsets and configured log or linear amplitude. Five bins are k-2 through k+2; three-bin mode retains the original closed form. Vertex delta=-b/(2*a), frequency=f[k]+delta*df; amplitude=q(delta) or exp(q(delta)). No fallback to fewer points. Reject tied maximum, unavailable neighbor, nonpositive/invalid values, nonconcavity, or vertex beyond half-bin. Native and padded spectra select their maxima independently.",
    "zero_padding": "nfft=pad_factor*N; refine sampled spectrum only. Report native_df=fs/N, padded_df=fs/nfft, Hann ENBW=fs*sum(w**2)/sum(w)**2; these are not interchangeable with physical resolution.",
    "raw_metrics": "raw_rms=sqrt(mean((raw-mean(raw))**2)); raw_rms_including_dc separate; raw_peak_to_peak=max(raw)-min(raw). Original retained samples, no trimming of outliers.",
    "noise": "Each stated noise control kept separate; full file by default. Same one-sided Hann periodogram PSD abs(FFT)**2/(fs*sum(w**2)), double interior positive bins. PSD in each sensor output unit squared/Hz (IM counts^2/Hz; accel g^2/Hz), not amplitude. Original noise PSD/RMS retained. Supplemental nominal-band SNR uses the explicitly configured independent controls; no detection limit, responsivity or sensitivity calculated.",
    "quality": "Boundary coverage failures always suppress FFT. Internal gap > max_gap_over_median remains a strict gap-check failure; default diagnostic mode also computes the same explicitly flagged linearly interpolated FFT. strict_global_* columns retain the strict result; fft_computed is computational availability, fft_valid is legacy strict gap eligibility, neither guarantees calibrated accuracy. Ratio comparison tolerance=max(1e-9,abs(threshold)*1e-9) handles exact-threshold floating error. Gap count, duration sum and duration fraction recorded; these do not assert lost samples. Raw extrema occupancy is diagnostic only and does not establish ADC/sensor saturation. Jitter, cycles, sampling margin, excursions and target mismatch remain flagged.",
    "sensitivity": "Additional symmetric sweep trims and raw-order FFT at the same estimated fs; diagnostics of window and interpolation dependence, not independent repeats or alternative primary results.",
    "resolution": "Independent record-level estimates grouped by base,target,window,method; arithmetic mean of finite estimates, sample SD(ddof=1) only n>=2. Computed and strict-gap views saved separately; flagged finite estimates retained in computed view with strict counts. Differences of group means are descriptive, not synchronized or calibrated two-tone resolution.",
    "target_diagnostics": "Supplement only: maximum bin inside target +/- min(0.4*target,max(3*native_df,0.15*target)), unless explicit halfwidth. Minimum0.1Hz by default. Native/padded fits using the configured interpolation_points match selected bins; no replacement if boundary/nonlocal/invalid. Rank interpolated interior local maxima above minimum and below Nyquist; all noise/sidelobes retained, exact ties share rank. Rankings and relative amplitude are not SNR. Primary global peaks never change.",
    "volume": "Selected files, target frequencies, levels, windows and pairing confirmation are recorded in effective CONFIG. Plot numeric volume ascending10to15; original segment order preserved. Each primary window is split into contiguous configured-duration subwindows (defaults: central20s, subwindows5s; effective values in parameters.json); not repeats. FFT amplitude, raw RMS, raw peak-to-peak remain distinct.",
    "volume_reference": "Explicit file pairs joined by volume only. Normalize each sensor to its own configured reference-level amplitude (default volume10). Plot IM amplitude vs accel amplitude and Data28 counts/g ratio; not calibrated responsivity or a synchronized transfer function. Preserve all points and mark accel repeated-extrema occupancy at or above the configured threshold (default1%); usable descriptive ratio excludes flagged plateaus, unavailable FFT/amplitude, unequal targets or incompatible dominant frequencies. Timing warnings remain. User confirmation records condition matching, not simultaneous sampling.",
    "cross_sensor": "User-confirmed condition-matched volume references are compared at whole-window level; no synchronized phases/coherence/transfer function, no pulse-to-pulse ratios, no physical calibration. Accelerometer sweep disabled by request.",
    "snr": "Supplemental noise-control-referenced nominal-band excess-power SNR; default nominal +/-1Hz. Excited native Hann PSD integrated in the same exact physical band as the separate control's Welch PSD, clipped to common represented frequency support. Noise Welch: configured6s periodicHann,50%overlap,constant detrend,linear-power mean,no padding; too-short/unavailable controls not silently shortened. Piecewise-linear trapezoidal PSD integration including boundary interpolation; DC endpoint retained after detrending when the band reaches0Hz. Pexcess=Pexcited-Pnoise; SNRlinear=Pexcess/Pnoise, SNRdB=10log10(SNRlinear). Nonpositive excess gives NaN SNR and explicit status; measured Pexcited/Pnoise retained separately. No cross-control averaging, errorbars or automatic detection claim. Assumes independent control represents additive uncorrelated background during driven recording; IM_1 excess includes amplifier-added noise. Nearby interference also contributes. Original global FFT selection is unchanged. All effective windows/bands/df/ENBW/segment counts and timing flags recorded in SNR CSV and parameters.json. Reference: https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.welch.html",
    "selected_view": "Additional sweep/SNR plots use plots.sweep_view (default>=10Hz), selecting already-computed rows by nominal frequency. Complete plots and original acquisition time mapping remain; no recalculation or alternative peak search.",
    "figures": "Main spectrogram restored to linear frequency0..Nyquist, viridis,100dB below max,2s Hann75%overlap; configurable. Cyan dots at nominal segment midpoints annotate desired frequencies, not peak estimates. Both linear/log frequency-axis versions use identical PSD and color limits. RMS/peak-to-peak amplitude panels use log axes; global-vs-target amplitudes explicitly distinguished. Resolution per-record spectra normalized to each own maximum INSIDE display crop for shape comparison only; numerical amplitudes unchanged. Resolution categorical method x-axis and mHz offset from base expose every repeat. Interpolation demo uses actual global-maximum native bin plus the configured symmetric neighborhood (default one neighbor on each side). Volume plots sort numeric levels ascending, never shift segment mapping. Plot settings below describe defaults; effective plot/FFT settings are recorded in parameters.json. Raw min/max envelopes in time bins of width max(0.02s,record_span/6000); spectrogram linearly interpolates N points between first/last record timestamps, fs=(N-1)/span, 2s periodic Hann (capped to record), configured75% overlap, constant detrend, one-sided PSD, dB relative to 1 unit^2/Hz, color range100dB below max. Spectrogram across internal gaps is explicitly labeled diagnostic by default; strict mode omits it. Dense spectrum displays keep the exact maximum in each of at most 6000 contiguous log-index buckets; peak selection and all numerical estimates use every FFT bin. Volume waveform figures show central three nominal cycles (capped at 2s), raw samples unchanged. Full positive spectra for each window; resolution zoom changes display only.",
}
METHODS.update(COMPARISON_METHODS)
METHODS.update(RELATIVE_SUMMARY_METHODS)
LIMITATIONS = [
    "Original-time sine fitting assumes Time_us represents acquisition instants; the timestamp source is unconfirmed. A good fit cannot independently calibrate timestamp scale or excitation frequency. The stationary single-sine model can be affected by harmonics/drift, and residual RMS is not a measured noise floor.",
    "Five-point quadratic fits are not guaranteed more accurate than three-point fits. Native Hann outer bins can approach main-lobe zeros and bias log-domain frequency/amplitude fits; padded bins span a narrower interval. Neither fit point count nor padding increases physical resolving power.",
    "Nominal-band SNR uses independent noise controls; it depends on bandwidth and background representativeness. Amplifier-added noise contributes relative to the nothing-on control. Interference in the band is included; excess power is not proof of a detected sine fundamental.",
    "Start offsets and transition times are configured assumptions until verified against experiment notes; nominal dwell durations do not establish actual switching times.",
    "Global maximum can be interference or a harmonic; neither a small frequency error nor a large peak establishes detection or calibrated bandwidth.",
    "Computed FFTs with internal gap warnings are diagnostics; strict-gap results are retained separately. Neither view establishes clock accuracy. Timestamp jitter, gaps, interpolation, finite window length and low samples/cycle can bias frequency/amplitude; analog sensor/logger/drive bandwidth and clock accuracy remain uncalibrated.",
    "Generator 1 Vpp and amplifier volume are settings, not measured physical input. IM amplitudes remain Data28 counts; accel amplitudes use nominally converted g.",
    "Repeat counts follow effective user-specified FILES and are saved separately. Sample SD uses only settings with at least two finite independent-record estimates; small n does not establish a resolution limit. Separate single-tone estimates do not measure simultaneous two-tone resolution. Between-batch conditions and clock drift remain unverified.",
    "Selected volume pair and matching-condition confirmation follow effective CONFIG; acquisitions are sequential. Condition-level Data28 counts/g ratios can include drift and are not calibrated sensitivity. Repeated accel extrema/plateaus limit reference amplitudes; flagged points remain visible.",
]


def _save_figure(run, figures, fig, name):
    from paper_figure_style import save_paper_figure
    save_paper_figure(run, fig, name, figures, dpi=150)


def _input_hash(path):
    digest = sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _snapshot_live_cells(run):
    try:
        shell = get_ipython()
    except NameError:
        shell = None
    history = [] if shell is None else shell.history_manager.input_hist_raw
    selected = [text for text in history if text.lstrip().startswith("# sinewave-cell:")]
    note = {"available": bool(selected), "source": "IPython execution history, sinewave-cell markers only",
            "note": "Disk notebook also archived. History contains execution order and repeated cell executions; latest definitions apply."}
    if selected:
        text = "\n\n# ---- next executed notebook cell ----\n\n".join(selected) + "\n"
        saved = run.write_text("scripts/executed_notebook_cells.py", text)
        note.update(snapshot_path=str(saved.relative_to(run.path)), sha256=sha256(text.encode("utf-8")).hexdigest())
    run.write_json("notebook_execution_source.json", note)


def _small_resolution_spectrum(row, spectrum):
    # Keep all original points inside display zoom; full-band plots are already saved.
    halfwidth = max(0.06, 6 * row["native_df_hz"])
    out = {}
    for freq_key, amp_key in (("f", "amplitude"), ("padded_f", "padded_amplitude")):
        f = spectrum[freq_key]
        select = abs(f - row["base_hz"]) <= halfwidth
        out[freq_key], out[amp_key] = f[select].copy(), spectrum[amp_key][select].copy()
    return out


def run_sine_analysis(config, results_root=RESULTS_ROOT, notebook_path=NOTEBOOK_PATH, execution_script_paths=None):
    effective = deepcopy(config)
    effective["fft"].setdefault("interpolation_points", 3)
    effective["sine_fit"] = sine_fit_settings(effective.get("sine_fit"))
    relative_method = effective["plots"].setdefault("relative_frequency_method", "global_padded_3point")
    if relative_method not in [name for name, _ in RESOLUTION_METHODS]:
        raise ValueError("Unknown plots.relative_frequency_method")
    desired_repeats = effective["resolution"].get("desired_records_per_setting", 3)
    if isinstance(desired_repeats, bool) or not isinstance(desired_repeats, int) or desired_repeats < 1:
        raise ValueError("resolution.desired_records_per_setting must be a positive integer")
    plan = build_plan(effective)
    selected = plan[plan["selected"]].copy()
    inputs = []
    for (sensor, path), group in selected.groupby(["sensor", "path"], sort=False):
        inputs.append({"path": path, "sensor": sensor, "role": sorted(group["role"].unique().tolist()),
                       "window_count": len(group)})
    parameters = {"config": effective, "schedule_presets_hz": deepcopy(SCHEDULE_PRESETS),
                  "methods": METHODS, "volume_reference_methods": VOLUME_REFERENCE_METHODS, "snr_methods": SNR_METHODS, "complete_schedule": plan.to_dict("records"),
                  "selected_schedule": selected.to_dict("records"),
                  "documented_generator_amplitude_vpp": 1.0,
                  "documented_generator_amplitude_scope": "Prior experimental context only; unverified for added IM_39-45. This setting is not used in frequency estimation.",
                  "selected_files": deepcopy(FILES),
                  "conversions": {"Time_us_to_s": 1e-6, "IM_Data28_scale": 1.0,
                      "accel_output_unit": "g", "accel_config": deepcopy(effective["accel_conversion"]),
                      "accel_per_file": [dict(path=item["path"], full_scale_g=accel_conversion_for(item["path"], effective["accel_conversion"])[0],
                                              g_per_count=accel_conversion_for(item["path"], effective["accel_conversion"])[1])
                                         for item in inputs if item["sensor"] == "accel"]},
                  "fixed_quality_defaults": {"minimum_samples_to_compute": 4, "small_sample_warning_below": 8,
                      "few_actual_cycles_below": 5, "few_samples_per_cycle_below": 10,
                      "severe_samples_per_cycle_below": 4, "near_nyquist_fraction": 0.9,
                      "crest_factor_warning_above": 10.0, "robust_z_warning_above": 20.0,
                      "gap_ratio_comparison_tolerance": "max(1e-9,abs(threshold)*1e-9)",
                      "global_target_mismatch_tolerance": "max(2*native_df_hz, 0.05*target_hz)",
                      "raw_order_difference_warning_bins": 0.1, "near_tie_rtol": 1e-12, "near_tie_atol": 0.0},
                  "results_root_source": config_source(),
                  "limitations": LIMITATIONS}
    script_paths = source_paths(Path(notebook_path), extras=(execution_script_paths or ()))
    rows, sensitivity_rows, audits, figures, hashes = [], [], [], [], []
    volume_subwindows = []
    snr_table = pd.DataFrame()
    estimator_table = pd.DataFrame()
    relative_records, relative_groups, repeat_counts = (pd.DataFrame() for _ in range(3))
    snr_sweep_items, snr_noise_references, selected_view_rows = [], [], []
    noise_items = {sensor: [] for sensor in effective["sensors"]}
    resolution_items = {}
    with AnalysisRun(results_root, "resolution_analysis_sinewave", parameters, inputs,
                     script_paths, repository_path=PROJECT) as run:
        print("Output directory:", run.path, flush=True)
        run.write_text("methods.txt", "\n\n".join(f"{name}: {text}" for name, text in METHODS.items()) + "\n")
        run.write_json("limitations.json", LIMITATIONS)
        _snapshot_live_cells(run)
        plan.to_csv(run.path / "schedule.csv", index=False)
        repeat_rows = []
        for key, part in selected[selected["role"] == "resolution"].drop_duplicates(
                ["sensor", "path", "base_hz", "target_hz"]).groupby(["sensor", "base_hz", "target_hz"], sort=True):
            repeat_rows.append({"sensor": key[0], "base_hz": key[1], "target_hz": key[2],
                "configured_records": len(part), "desired_records": desired_repeats,
                "records_still_needed": max(0, desired_repeats - len(part)),
                "files": ";".join(Path(path).name for path in part["path"])})
        repeat_counts = pd.DataFrame(repeat_rows)
        repeat_counts.to_csv(run.path / "resolution_repeat_counts.csv", index=False)
        try:
            # One record at a time; all reads leave original files unchanged.
            for file_number, ((sensor, path), file_plan) in enumerate(selected.groupby(["sensor", "path"], sort=False), 1):
                record = load_record(path, sensor, effective["accel_conversion"])
                audit = {"sensor": sensor, "path": path, "unit": record["unit"], "full_scale_g": record["metadata"]["nominal_full_scale_g"],
                         "signal_scale": record["metadata"]["signal_scale"], **record["metadata"]["timestamp_audit"]}
                audits.append(audit)
                pd.DataFrame(audits).to_csv(run.path / "timestamp_audit.csv", index=False)
                hashes.append({"path": path, "sha256": _input_hash(path), "size_bytes": Path(path).stat().st_size})
                run.write_json("input_hashes.json", hashes)
                prefix = f"{file_number:02d}_{sensor}_{Path(path).stem}"
                file_rows, items, peak_catalogs = [], [], []
                for segment in file_plan.to_dict("records"):
                    end = segment["analysis_end_s"]
                    if end is None or pd.isna(end):
                        end = record["t"][-1] + audit["dt_mean_s"]
                    row, spectrum = analyze_segment(record, segment["analysis_start_s"], end,
                                                    segment["target_hz"], effective["fft"])
                    row = {**segment, **row, "unit": record["unit"], "full_scale_g": record["metadata"]["nominal_full_scale_g"],
                           "signal_scale": record["metadata"]["signal_scale"], "file_id": segment["file_label"],
                           "label": segment["file_label"] + (" / " + str(segment["noise_condition"]) if segment["role"] == "noise" else "")}
                    if segment["role"] == "sweep" and effective["target_diagnostics"]["enabled"]:
                        target_metrics, catalog = target_peak_diagnostics(row, spectrum, effective["target_diagnostics"])
                        row.update(target_metrics)
                        if not catalog.empty:
                            catalog.insert(0, "target_hz", row["target_hz"])
                            catalog.insert(0, "segment_index", row["segment_index"])
                            catalog.insert(0, "file_label", row["file_label"])
                            peak_catalogs.append(catalog)
                    if segment["role"] == "resolution":
                        row.update(add_frequency_estimators(row, spectrum, effective["sine_fit"], _global_peak))
                        print(f"  {row['file_label']} / {row['window_variant']}: original-time sine fit "
                              f"{row['raw_timestamp_sine_status']}", flush=True)
                    rows.append(row)
                    file_rows.append(row)
                    items.append((row, spectrum))
                    pd.DataFrame(rows).to_csv(run.path / "window_results.csv", index=False)
                    if effective["snr"]["enabled"] and sensor in effective["snr"]["sensors"]:
                        if segment["role"] == "noise" and row["file_label"] in effective["snr"]["noise_files"].get(sensor, []):
                            reference = prepare_snr_noise(row, spectrum, effective["snr"])
                            snr_noise_references.append(reference)
                            pd.DataFrame({"frequency_hz": reference["f"], "psd_signal_units_squared_per_hz": reference["psd"], "signal_unit": record["unit"]}).to_csv(
                                run.path / (prefix + "_snr_noise_welch_psd.csv"), index=False)
                            run.write_json("snr_noise_references.json", [
                                {k: v for k, v in ref.items() if k not in ("f", "psd")} for ref in snr_noise_references])
                        if segment["role"] == "sweep":
                            snr_sweep_items.append((row, {key: spectrum[key] for key in ("f", "psd")}))
                    if segment["role"] == "noise":
                        noise_items[sensor].append((row, {k: spectrum[k] for k in ("f", "amplitude", "psd")}))
                    if segment["role"] == "resolution":
                        resolution_items.setdefault(segment["window_variant"], []).append((row, _small_resolution_spectrum(row, spectrum)))
                    if segment["role"] == "resolution":
                        cropped = _small_resolution_spectrum(row, spectrum)
                        spectral_rows = []
                        for grid, fk, ak in (("native", "f", "amplitude"), ("padded", "padded_f", "padded_amplitude")):
                            spectral_rows.append(pd.DataFrame({"grid": grid, "frequency_hz": cropped[fk], "peak_amplitude": cropped[ak], "unit": record["unit"]}))
                        pd.concat(spectral_rows, ignore_index=True).to_csv(run.path / f"{prefix}_{row['window_variant']}_spectrum.csv", index=False)
                        if (row["window_variant"] == next(iter(effective["resolution"]["windows_s"]))
                                and row["file_label"] in effective["plots"]["interpolation_example_files"]):
                            _save_figure(run, figures, plot_interpolation_demo(row, spectrum), prefix + "_interpolation_demo")
                    if segment["role"] == "volume":
                        width = effective["volume_subwindow_s"]
                        a, block_index = row["analysis_start_s"], 1
                        while a + width <= row["analysis_end_s"] + 1e-9:
                            block, _ = analyze_segment(record, a, a + width, row["target_hz"], effective["fft"])
                            block.update(sensor=sensor, file_label=row["file_label"], path=path, role="volume_subwindow",
                                unit=record["unit"], volume_setting=row["volume_setting"], subwindow_index=block_index,
                                interpretation="within-record variation; not independent repeats or synchronized sensor pairs")
                            volume_subwindows.append(block)
                            a += width
                            block_index += 1
                        pd.DataFrame(volume_subwindows).to_csv(run.path / "volume_subwindow_diagnostics.csv", index=False)
                    if segment["role"] == "sweep":
                        for trim in effective["sensitivity_trims_s"]:
                            a, b = segment["nominal_start_s"] + trim, segment["nominal_end_s"] - trim
                            diagnostic = {**segment, "symmetric_trim_s": trim,
                                          "analysis_start_s": a, "analysis_end_s": b, "unit": record["unit"]}
                            try:
                                check, _ = analyze_segment(record, a, b, segment["target_hz"], effective["fft"])
                                diagnostic.update(check, status="computed", reason="")
                            except ValueError as exc:
                                diagnostic.update(status="unsupported", reason=str(exc))
                            sensitivity_rows.append(diagnostic)
                        pd.DataFrame(sensitivity_rows).to_csv(run.path / "window_sensitivity.csv", index=False)
                frame = pd.DataFrame(file_rows)
                if peak_catalogs:
                    pd.concat(peak_catalogs, ignore_index=True).to_csv(run.path / (prefix + "_peak_catalog.csv"), index=False)
                _save_figure(run, figures, plot_record_overview(record, frame, Path(path).stem,
                    max_gap_over_median=effective["fft"]["max_gap_over_median"],
                    gap_policy=effective["fft"].get("gap_policy", "diagnostic")), prefix + "_record_overview")
                for page, fig in enumerate(plot_spectra(items, Path(path).stem + ": full positive spectra",
                                                       effective["plots"]["spectra_per_page"]), 1):
                    _save_figure(run, figures, fig, prefix + f"_spectra_{page:02d}")
                if frame["role"].eq("sweep").any():
                    subset = frame[frame["role"] == "sweep"]
                    complete_schedule = plan[(plan["sensor"] == sensor) & (plan["path"] == path) & (plan["role"] == "sweep")]
                    for frequency_scale in effective["plots"]["spectrogram_frequency_scales"]:
                        display_config = deepcopy(effective)
                        display_config["plots"]["spectrogram_frequency_scale"] = frequency_scale
                        _save_figure(run, figures, plot_spectrogram(record, complete_schedule, Path(path).stem, display_config),
                                     prefix + f"_spectrogram_{frequency_scale}")
                    _save_figure(run, figures, plot_sweep(subset, f"{Path(path).stem}: frequency estimates", effective["plots"]), prefix + "_sweep")
                    _save_figure(run, figures, plot_sweep_amplitudes(subset, f"{Path(path).stem}: amplitude metrics"), prefix + "_amplitudes")
                    if effective["plots"]["sweep_view"]["enabled"]:
                        view = select_sweep_view(subset, effective["plots"]["sweep_view"])
                        selected_view_rows.extend(view.to_dict("records"))
                        view_title = f"{Path(path).stem}: {sweep_view_label(effective['plots']['sweep_view'])}"
                        _save_figure(run, figures, plot_sweep(view, view_title, effective["plots"]), prefix + "_selected_sweep")
                        _save_figure(run, figures, plot_sweep_amplitudes(view, view_title), prefix + "_selected_amplitudes")
                    if effective["target_diagnostics"]["enabled"]:
                        _save_figure(run, figures, plot_peak_ranking(subset, f"{Path(path).stem}: target-peak ranks"), prefix + "_peak_ranks")
                if frame["role"].eq("volume").any():
                    subset = frame[frame["role"] == "volume"]
                    blocks = pd.DataFrame([b for b in volume_subwindows if b["path"] == path and b["sensor"] == sensor])
                    _save_figure(run, figures, plot_volume(subset, f"{Path(path).stem}: volume response", blocks,
                        plateau_fraction_threshold=effective["volume_reference"]["plateau_fraction_threshold"]), prefix + "_volume")
                volume_items = [(r, s) for r, s in items if r["role"] == "volume"]
                if volume_items:
                    _save_figure(run, figures, plot_volume_waveforms(volume_items, f"{Path(path).stem}: raw waveform examples"), prefix + "_volume_waveforms")
                print(f"{sensor} / {Path(path).name}: {len(file_rows)} windows", flush=True)
            results = pd.DataFrame(rows)
            for role in (role for role, enabled in effective["sections"].items() if enabled):
                results[results["role"] == role].to_csv(run.path / (role + "_summary.csv"), index=False)
            for sensor, items in noise_items.items():
                if items:
                    _save_figure(run, figures, plot_noise(items, f"{sensor}: separate noise controls"), sensor + "_noise_comparison")
            if effective["sections"]["sweep"]:
                pd.DataFrame(selected_view_rows).to_csv(run.path / "selected_sweep_summary.csv", index=False)
            if effective["snr"]["enabled"]:
                snr_table = build_sweep_snr(snr_sweep_items, snr_noise_references, effective["snr"])
                snr_table.to_csv(run.path / "snr_summary.csv", index=False)
            if not snr_table.empty:
                for (sensor, sweep_path), snr_group in snr_table.groupby(["sensor", "path"], sort=False):
                    snr_prefix = f"{sensor}_{Path(sweep_path).stem}"
                    _save_figure(run, figures, plot_sweep_snr(snr_group, f"{Path(sweep_path).stem}: frequency-range SNR"), snr_prefix + "_snr")
                    if effective["plots"]["sweep_view"]["enabled"]:
                        snr_view = select_sweep_view(snr_group, effective["plots"]["sweep_view"])
                        _save_figure(run, figures, plot_sweep_snr(snr_view,
                            f"{Path(sweep_path).stem}: SNR / {sweep_view_label(effective['plots']['sweep_view'])}"), snr_prefix + "_selected_snr")
                if effective["plots"]["sweep_view"]["enabled"]:
                    select_sweep_view(snr_table, effective["plots"]["sweep_view"]).to_csv(run.path / "selected_snr_summary.csv", index=False)
            groups, comparisons, strict_groups, strict_comparisons = (pd.DataFrame() for _ in range(4))
            if effective["sections"]["resolution"]:
                resolution = results[results["role"] == "resolution"]
                estimator_table = frequency_estimator_table(resolution)
                estimator_table.to_csv(run.path / "frequency_estimator_comparison.csv", index=False)
                relative_records, relative_groups = build_relative_summary(estimator_table)
                relative_records.to_csv(run.path / "resolution_relative_records.csv", index=False)
                relative_groups.to_csv(run.path / "resolution_relative_groups.csv", index=False)
                diagnostic_columns = [key for key in resolution.columns if key.startswith('raw_timestamp_sine_')]
                resolution[["file_label", "path", "target_hz", "window_variant", "analysis_start_s", "analysis_end_s",
                            "sample_count", "sine_fit_timestamp_basis", "sine_minus_padded_3point_hz",
                            "padded_5point_minus_3point_hz", "native_5point_minus_3point_hz", *diagnostic_columns]].to_csv(
                    run.path / "sine_fit_diagnostics.csv", index=False)
                groups, comparisons = summarize_resolution(resolution)
                strict_groups, strict_comparisons = summarize_resolution(resolution, "strict_gap")
                strict_groups.to_csv(run.path / "resolution_groups_strict_gap.csv", index=False)
                strict_comparisons.to_csv(run.path / "resolution_comparisons_strict_gap.csv", index=False)
                groups.to_csv(run.path / "resolution_groups.csv", index=False)
                comparisons.to_csv(run.path / "resolution_comparisons.csv", index=False)
                for variant, resolution_spectra in resolution_items.items():
                    _save_figure(run, figures, plot_relative_frequency(relative_records, relative_groups, variant,
                                 method=relative_method), f"resolution_{variant}_relative_primary")
                    _save_figure(run, figures, plot_absolute_ppm(relative_records, relative_groups, variant,
                                 method=relative_method), f"resolution_{variant}_absolute_ppm_primary")
                    _save_figure(run, figures, plot_estimator_errors(estimator_table, variant), f"resolution_{variant}_estimator_errors")
                    _save_figure(run, figures, plot_sine_fit_agreement(resolution, variant), f"resolution_{variant}_method_agreement")
                    for index, fig in enumerate(plot_resolution_comparison(resolution_spectra), 1):
                        _save_figure(run, figures, fig, f"resolution_{variant}_comparison_{index:02d}")
            volume_reference = (build_volume_reference(results, effective["volume_reference"])
                                if results["role"].eq("volume").any() else pd.DataFrame())
            if effective["sections"]["volume"]:
                volume_reference.to_csv(run.path / "volume_reference.csv", index=False)
            if not volume_reference.empty:
                reference_figures = plot_volume_reference(volume_reference)
                if not isinstance(reference_figures, (list, tuple)):
                    reference_figures = [reference_figures]
                for index, fig in enumerate(reference_figures, 1):
                    _save_figure(run, figures, fig, f"volume_reference_{index:02d}")
            summary = {"input_files": len(inputs), "analyzed_windows": len(results),
                       "fft_computed_windows": int(results["fft_computed"].sum()),
                       "fft_strict_gap_valid_windows": int(results["fft_valid"].sum()),
                       "diagnostic_timing_warning_windows": int(results["analysis_status"].eq("computed_with_timing_warning").sum()),
                       "quality_flagged_windows": int(results["quality_flags"].ne("").sum()),
                       "original_time_sine_fit_available_windows": int(np.isfinite(results.get("raw_timestamp_sine_hz", pd.Series(dtype=float))).sum()),
                       "primary_frequency_estimator": "global_padded_3point",
                       "relative_figure_method": relative_method,
                       "records_needed_for_planned_repeats": int(repeat_counts["records_still_needed"].sum()) if not repeat_counts.empty else 0,
                       "figure_count": len(figures), "snr_comparisons": len(snr_table),
                       "finite_snr_db": int(np.isfinite(snr_table["snr_db"]).sum()) if not snr_table.empty else 0,
                       "limitations": LIMITATIONS}
            run.write_json("summary.json", summary)
            run.finish(summary)
        finally:
            # Actual windows and audit survive a later file/plot failure.
            run.update_parameters({"actual_windows": rows, "timestamp_audits": audits,
                                   "alternative_windows": sensitivity_rows, "volume_subwindows": volume_subwindows,
                                   "snr_comparisons": snr_table.to_dict("records"),
                                   "snr_noise_reference_metadata": [{k: v for k, v in ref.items() if k not in ("f", "psd")}
                                                                    for ref in snr_noise_references]})
    return {"output_path": str(run.path), "summary": results, "audit": pd.DataFrame(audits),
            "schedule": plan, "sensitivity": pd.DataFrame(sensitivity_rows), "figures": figures,
            "volume_subwindows": pd.DataFrame(volume_subwindows), "volume_reference": volume_reference,
            "snr": snr_table, "selected_sweep_view": pd.DataFrame(selected_view_rows),
            "frequency_estimator_comparison": estimator_table,
            "relative_records": relative_records, "relative_groups": relative_groups, "repeat_counts": repeat_counts,
            "resolution_groups": groups, "resolution_comparisons": comparisons,
            "resolution_groups_strict_gap": strict_groups, "resolution_comparisons_strict_gap": strict_comparisons,
            "limitations": LIMITATIONS}


## Run analysis

This cell reads the selected recordings and creates a new dated run. Failures retain partial outputs and status; previous runs are not overwritten.


In [ ]:
# sinewave-cell: run_analysis
RESULTS = run_sine_analysis(CONFIG)
OUTPUT_PATH = Path(RESULTS["output_path"])
print("Completed:", OUTPUT_PATH)


## Results


In [ ]:
# sinewave-cell: display_helpers
def show_columns(table, columns):
    if table.empty:
        print("No selected records for this section.")
        return
    display(table[[key for key in columns if key in table]])


def show_figures(*keywords):
    for path in RESULTS["figures"]:
        if any(word in Path(path).stem for word in keywords):
            display(Image(filename=path))


def show_record_figures(role):
    selected = RESULTS["summary"].query("role == @role")[["sensor", "path"]].drop_duplicates()
    for row in selected.itertuples():
        show_figures(f"_{row.sensor}_{Path(row.path).stem}_record_overview")


FREQUENCY_COLUMNS = ["sensor", "file_label", "target_hz", "window_variant",
    "analysis_start_s", "analysis_end_s", "actual_start_s", "actual_end_s", "sample_count",
    "fs_hz", "native_df_hz", "padded_df_hz", "enbw_hz", "interpolation_points", "interpolation_scale", "analysis_status", "strict_gap_check_passed",
    *[name + "_hz" for name in ESTIMATORS], "global_interpolation_status",
    "global_padded_interpolation_status", "raw_timestamp_sine_status", "sine_minus_padded_3point_hz", "quality_flags"]
AMPLITUDE_COLUMNS = ["sensor", "file_label", "target_hz", "volume_setting", "unit",
    "raw_rms", "raw_peak_to_peak", "raw_min_value", "raw_max_value", "raw_extrema_occupancy_fraction", "analysis_status", "global_bin_amplitude", "global_parabolic_amplitude",
    "global_padded_amplitude", "global_padded_parabolic_amplitude", "quality_flags"]
display(RESULTS["audit"])


## Relative frequency and repeatability

Plots retain individual recordings and mean +/- sample SD for each base frequency. Baseline subtraction changes the display origin, not frequency calibration. Absolute nominal errors and all estimator comparisons remain separate. Each setting uses the three files in FILES; missing estimates are not filled.


In [ ]:
# sinewave-cell: resolution_results
primary_resolution_window = next(iter(CONFIG["resolution"]["windows_s"]))
show_figures(f"resolution_{primary_resolution_window}_relative_primary",
             f"resolution_{primary_resolution_window}_absolute_ppm_primary")
display(RESULTS["repeat_counts"])
relative_main = RESULTS["relative_groups"]
relative_main = relative_main[(relative_main["window_variant"] == primary_resolution_window)
                              & (relative_main["method"] == CONFIG["plots"]["relative_frequency_method"])]
show_columns(relative_main, ["sensor", "base_hz", "target_hz", "valid_count", "baseline_count",
    "mean_relative_mhz", "sample_sd_mhz", "mean_nominal_error_ppm", "baseline_mean_hz", "baseline_sd_hz"])
show_columns(RESULTS["summary"].query("role == 'resolution'"), FREQUENCY_COLUMNS)
display(RESULTS["resolution_comparisons"])
if CONFIG["plots"]["show_strict_gap_tables"]:
    show_columns(RESULTS["resolution_groups_strict_gap"], ["base_hz", "target_hz", "window_variant", "method", "n_valid", "mean_hz"])
show_figures(f"resolution_{primary_resolution_window}_comparison",
             f"resolution_{primary_resolution_window}_method_agreement", "_interpolation_demo")
# The trimmed_90s versions are saved separately; those windows are not extra repeats.


## Diagnostics and output files

Inspect strict-gap tables, full spectra, timestamp audits, interpolation examples and the 90 s window comparison. Repeated windows from one recording are not extra experiments. Arial is used when available, otherwise DejaVu Sans.


In [ ]:
# sinewave-cell: diagnostics
flagged = RESULTS["summary"][RESULTS["summary"]["quality_flags"].ne("")]
show_columns(flagged, ["sensor", "file_label", "target_hz", "window_variant", "fft_valid",
                       "fft_computed", "analysis_status", "dt_cv", "max_gap_over_median", "gap_interval_count", "gap_duration_fraction", "samples_per_target_cycle", "quality_flags"])
show_columns(RESULTS["sensitivity"], ["sensor", "file_label", "target_hz", "symmetric_trim_s",
    "analysis_start_s", "analysis_end_s", "global_parabolic_hz", "global_padded_parabolic_hz",
    "raw_order_parabolic_hz", "status", "reason", "quality_flags"])
# Each record overview and full spectrum is saved regardless of inline display choice.
# Use e.g. show_figures("02_IM_IM_16_record_overview") after checking figures.json for filenames.
if CONFIG["plots"]["show_all_spectra"]:
    show_figures("_spectra_", "_record_overview", "_zoom_")
print("Exact output path:", OUTPUT_PATH)
print("Windows:", len(RESULTS["summary"]), "| FFT computed:", int(RESULTS["summary"].fft_computed.sum()),
      "| strict gap comparison passes:", int(RESULTS["summary"].fft_valid.sum()))
for limitation in RESULTS["limitations"]:
    print("-", limitation)
print("CSV / PNG / editable SVG / parameters.json / inputs.json / provenance.json / methods.txt / run_status.json / scripts/")
